# NYC Parking Violations FY2025: Exploratory Analysis

This notebook profiles the parking violations dataset and builds a first borough GeoJSON map. The CSV has no latitude/longitude, so maps use aggregated borough counts rather than ticket points.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import urllib.request

import pandas as pd
import plotly.express as px

DATA_DIR = Path('.')
CSV_PATH = DATA_DIR / 'Parking_Violations_Issued_-_Fiscal_Year_2025.csv'
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

FY_START = pd.Timestamp('2024-07-01')
FY_END = pd.Timestamp('2025-06-30')
CHUNKSIZE = 500_000

print(CSV_PATH.exists(), f'{CSV_PATH.stat().st_size / 1_000_000:.1f} MB')

In [ ]:
BOROUGH_MAP = {
    'NY': 'Manhattan', 'MN': 'Manhattan', 'MAN': 'Manhattan', 'NEW YORK': 'Manhattan',
    'K': 'Brooklyn', 'BK': 'Brooklyn', 'KINGS': 'Brooklyn',
    'Q': 'Queens', 'QN': 'Queens', 'QNS': 'Queens', 'QUEEN': 'Queens', 'QUEENS': 'Queens',
    'BX': 'Bronx', 'BRONX': 'Bronx',
    'R': 'Staten Island', 'ST': 'Staten Island', 'RICH': 'Staten Island', 'STATEN ISLAND': 'Staten Island', 'RICHMOND': 'Staten Island',
}
CAMERA_CODES = {'7', '12', '36'}
BOROUGHS = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
WEEKDAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


def normalize_borough(x):
    if pd.isna(x):
        return 'Unknown'
    key = str(x).strip().upper()
    return BOROUGH_MAP.get(key, key.title() if key else 'Unknown')


def parse_hour(s):
    s = s.astype(str).str.strip().str.upper()
    h = pd.to_numeric(s.str[:2], errors='coerce')
    m = pd.to_numeric(s.str[2:4], errors='coerce')
    ap = s.str[4:5]
    valid = h.between(1, 12) & m.between(0, 59) & ap.isin(['A', 'P'])
    h24 = h.mask((ap == 'A') & (h == 12), 0).mask((ap == 'P') & (h < 12), h + 12)
    return h24.where(valid)

## Build Compact EDA Tables

The raw file is large, so this pass reads it in chunks and saves small CSV summaries in `outputs/eda/`.

In [ ]:
def build_eda_tables():
    out = OUTPUT_DIR / 'eda'
    out.mkdir(exist_ok=True)
    usecols = ['Issue Date', 'Violation Time', 'Violation Code', 'Violation Description',
               'Violation County', 'Violation Precinct', 'Registration State', 'Plate Type',
               'Issuing Agency', 'Vehicle Body Type', 'Vehicle Make']

    quality = Counter()
    borough = Counter(); borough_camera = Counter(); borough_curb = Counter()
    codes = Counter(); months = Counter(); hours = Counter(); weekdays = Counter(); hour_weekday = Counter()
    precincts = Counter(); states = Counter(); plates = Counter(); agencies = Counter(); makes = Counter()

    for chunk in pd.read_csv(CSV_PATH, usecols=usecols, dtype=str, chunksize=CHUNKSIZE, low_memory=False):
        quality['raw_rows'] += len(chunk)
        dt = pd.to_datetime(chunk['Issue Date'], errors='coerce', format='%m/%d/%Y')
        quality['invalid_dates'] += int(dt.isna().sum())
        in_fy = dt.between(FY_START, FY_END)
        quality['outside_fy'] += int((~in_fy & dt.notna()).sum())
        chunk = chunk.loc[in_fy].copy()
        dt = dt.loc[in_fy]

        chunk['borough'] = chunk['Violation County'].map(normalize_borough)
        chunk['code'] = chunk['Violation Code'].astype(str).str.strip()
        chunk['camera'] = chunk['code'].isin(CAMERA_CODES)
        chunk['hour'] = parse_hour(chunk['Violation Time'])
        chunk['weekday'] = dt.dt.day_name()
        chunk['month'] = dt.dt.to_period('M').astype(str)
        chunk['precinct'] = chunk['Violation Precinct'].astype(str).str.strip()

        quality['fy_rows'] += len(chunk)
        quality['precinct_zero'] += int(chunk['precinct'].eq('0').sum())
        quality['missing_borough'] += int(chunk['borough'].eq('Unknown').sum())
        quality['missing_hour'] += int(chunk['hour'].isna().sum())

        borough.update(chunk['borough']); borough_camera.update(chunk.loc[chunk['camera'], 'borough']); borough_curb.update(chunk.loc[~chunk['camera'], 'borough'])
        codes.update(zip(chunk['code'], chunk['Violation Description'].fillna('Unknown')))
        months.update(chunk['month']); hours.update(chunk['hour'].dropna().astype(int)); weekdays.update(chunk['weekday']); hour_weekday.update(zip(chunk['weekday'], chunk['hour'].dropna().astype(int)))
        precincts.update(chunk['precinct']); states.update(chunk['Registration State'].fillna('Unknown')); plates.update(chunk['Plate Type'].fillna('Unknown'))
        agencies.update(chunk['Issuing Agency'].fillna('Unknown')); makes.update(chunk['Vehicle Make'].fillna('Unknown'))

    borough_df = pd.DataFrame({'borough': sorted(set(borough) | set(borough_camera) | set(borough_curb))})
    borough_df['tickets'] = borough_df['borough'].map(borough).fillna(0).astype(int)
    borough_df['camera_tickets'] = borough_df['borough'].map(borough_camera).fillna(0).astype(int)
    borough_df['curbside_tickets'] = borough_df['borough'].map(borough_curb).fillna(0).astype(int)
    borough_df['camera_share'] = borough_df['camera_tickets'] / borough_df['tickets']

    pd.DataFrame([quality]).to_csv(out / 'quality.csv', index=False)
    borough_df.to_csv(out / 'borough.csv', index=False)
    pd.DataFrame([(a,b,c) for (a,b),c in codes.items()], columns=['code','description','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'codes.csv', index=False)
    pd.DataFrame(months.items(), columns=['month','tickets']).sort_values('month').to_csv(out / 'months.csv', index=False)
    pd.DataFrame(hours.items(), columns=['hour','tickets']).sort_values('hour').to_csv(out / 'hours.csv', index=False)
    pd.DataFrame(weekdays.items(), columns=['weekday','tickets']).to_csv(out / 'weekdays.csv', index=False)
    pd.DataFrame([(d,h,c) for (d,h),c in hour_weekday.items()], columns=['weekday','hour','tickets']).to_csv(out / 'hour_weekday.csv', index=False)
    pd.DataFrame(precincts.items(), columns=['precinct','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'precincts.csv', index=False)
    pd.DataFrame(states.items(), columns=['state','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'states.csv', index=False)
    pd.DataFrame(plates.items(), columns=['plate_type','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'plate_types.csv', index=False)
    pd.DataFrame(agencies.items(), columns=['agency','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'agencies.csv', index=False)
    pd.DataFrame(makes.items(), columns=['make','tickets']).sort_values('tickets', ascending=False).to_csv(out / 'makes.csv', index=False)

# Build summaries only if they do not already exist.
if not (OUTPUT_DIR / 'eda' / 'quality.csv').exists():
    build_eda_tables()

## Load Summaries

These tables are small and drive the charts below.

In [ ]:
EDA_DIR = OUTPUT_DIR / 'eda'
quality = pd.read_csv(EDA_DIR / 'quality.csv')
borough = pd.read_csv(EDA_DIR / 'borough.csv')
codes = pd.read_csv(EDA_DIR / 'codes.csv')
months = pd.read_csv(EDA_DIR / 'months.csv')
hours = pd.read_csv(EDA_DIR / 'hours.csv')
weekdays = pd.read_csv(EDA_DIR / 'weekdays.csv')
hour_weekday = pd.read_csv(EDA_DIR / 'hour_weekday.csv')
precincts = pd.read_csv(EDA_DIR / 'precincts.csv')
states = pd.read_csv(EDA_DIR / 'states.csv')
plates = pd.read_csv(EDA_DIR / 'plate_types.csv')
agencies = pd.read_csv(EDA_DIR / 'agencies.csv')
makes = pd.read_csv(EDA_DIR / 'makes.csv')

quality

## First Findings

Start with volume, violation types, borough distribution, and timing. We separate camera-like codes because they dominate the data and often have `Violation Precinct = 0`.

In [ ]:
codes.head(12)

In [ ]:
fig = px.bar(codes.head(12).sort_values('tickets'), x='tickets', y='description', orientation='h',
             title='Top violation descriptions in FY2025')
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

In [ ]:
plot_borough = borough[borough['borough'].isin(BOROUGHS)].sort_values('tickets', ascending=False)
fig = px.bar(plot_borough, x='borough', y=['curbside_tickets', 'camera_tickets'],
             title='Tickets by borough: curbside vs camera-like', labels={'value':'Tickets', 'variable':'Type'})
fig

In [ ]:
fig = px.line(months, x='month', y='tickets', markers=True, title='Monthly ticket volume across FY2025')
fig.update_layout(xaxis_title='', yaxis_title='Tickets')
fig

In [ ]:
weekday_plot = weekdays.set_index('weekday').reindex(WEEKDAYS).reset_index()
fig = px.bar(weekday_plot, x='weekday', y='tickets', title='Tickets by weekday')
fig.update_layout(xaxis_title='', yaxis_title='Tickets')
fig

In [ ]:
heat = hour_weekday.pivot(index='weekday', columns='hour', values='tickets').reindex(WEEKDAYS)
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Viridis',
                title='Ticket timing: weekday by hour')
fig.update_layout(xaxis_title='Hour of day', yaxis_title='')
fig

In [ ]:
fig = px.bar(states.head(10).sort_values('tickets'), x='tickets', y='state', orientation='h',
             title='Top registration states')
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

## Borough GeoJSON Map

This choropleth maps curbside tickets by borough. The GeoJSON join key is `properties.BoroName`.

In [ ]:
BOROUGH_GEOJSON_URL = 'https://raw.githubusercontent.com/nycehs/NYC_geography/master/borough.geo.json'
BOROUGH_GEOJSON_PATH = OUTPUT_DIR / 'borough.geo.json'

if not BOROUGH_GEOJSON_PATH.exists():
    urllib.request.urlretrieve(BOROUGH_GEOJSON_URL, BOROUGH_GEOJSON_PATH)

with open(BOROUGH_GEOJSON_PATH) as f:
    borough_geojson = json.load(f)

map_data = borough[borough['borough'].isin(BOROUGHS)]
fig = px.choropleth_map(map_data, geojson=borough_geojson, locations='borough',
                        featureidkey='properties.BoroName', color='curbside_tickets',
                        hover_name='borough', hover_data={'tickets':':,', 'camera_tickets':':,', 'curbside_tickets':':,', 'camera_share':':.1%'},
                        color_continuous_scale='Viridis', map_style='carto-positron',
                        center={'lat': 40.7128, 'lon': -74.0060}, zoom=9, opacity=0.82,
                        title='FY2025 curbside parking violations by borough')
fig.update_traces(marker_line_width=1.2, marker_line_color='white')
fig.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig.write_html(OUTPUT_DIR / 'eda_borough_curbside_map.html')
fig

## Draft-Aligned Analysis

The traffic draft frames the project as a public data story: scale, timing, geography, who gets fined, and policy limits. These cells add those angles without expanding the notebook too much.

In [ ]:
# Scale: the draft's "billion ticket machine" framing starts with total volume and camera share.
scale = quality.assign(
    precinct_zero_share = quality['precinct_zero'] / quality['fy_rows'],
    invalid_date_share = quality['invalid_dates'] / quality['raw_rows']
)
scale.T

In [ ]:
# Estimated fines by violation code.
# This uses the data dictionary's fine table. It is approximate because exact fine zone is not in our summary.
fines = pd.read_csv(EDA_DIR / 'fine_estimates.csv', dtype={'code': str})
fines[['code', 'violation_description_dict', 'tickets', 'fine_other_areas', 'estimated_fines_other_areas']].head(12)

In [ ]:
fig = px.bar(
    fines.head(10).sort_values('estimated_fines_other_areas'),
    x='estimated_fines_other_areas', y='violation_description_dict', orientation='h',
    title='Top violation codes by estimated fine value',
    labels={'estimated_fines_other_areas': 'Estimated fines ($)', 'violation_description_dict': ''}
)
fig

In [ ]:
# Geography: precinct 0 is mostly non-mappable camera-style enforcement, so rank real precincts separately.
precinct_plot = precincts[precincts['precinct'].astype(str) != '0'].head(15).sort_values('tickets')
fig = px.bar(precinct_plot, x='tickets', y='precinct', orientation='h',
             title='Top non-zero violation precincts')
fig.update_layout(yaxis_title='Precinct', xaxis_title='Tickets')
fig

In [ ]:
# Who gets fined: registration state and plate type give a first vehicle/owner profile.
plate_plot = plates.head(10).sort_values('tickets')
fig = px.bar(plate_plot, x='tickets', y='plate_type', orientation='h',
             title='Top plate types')
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

In [ ]:
# Enforcement channel: issuing agency helps separate traffic agents, cameras, police, and other issuers.
agencies.head(10)

### Policy and Equity Caveat

The draft asks whether enforcement is equitable across neighborhoods. This dataset alone cannot answer that causally. For a responsible equity analysis, we would need denominators such as registered vehicles, population, poverty, commute flows, curb space, meters, school-zone cameras, or precinct/community-district boundaries. Until then, raw ticket counts show enforcement volume, not individual risk or fairness.

## Investigating The Next Questions

The draft and first EDA point to two practical questions: which violations define each borough, and what does `Violation Precinct = 0` actually represent? These matter before making any precinct map.

In [ ]:
code_by_borough = pd.read_csv(EDA_DIR / 'code_by_borough.csv', dtype={'code': str})
precinct_zero_codes = pd.read_csv(EDA_DIR / 'precinct_zero_codes.csv', dtype={'code': str})
nonzero_precinct_codes = pd.read_csv(EDA_DIR / 'nonzero_precinct_codes.csv', dtype={'code': str})
channel_by_borough = pd.read_csv(EDA_DIR / 'channel_by_borough.csv')

code_by_borough.head()

In [ ]:
# Top violation types in each borough.
top_borough_codes = (
    code_by_borough[code_by_borough['borough'].isin(BOROUGHS)]
    .sort_values(['borough', 'tickets'], ascending=[True, False])
    .groupby('borough')
    .head(5)
)

top_borough_codes[['borough', 'code', 'description', 'tickets']]

In [ ]:
fig = px.bar(
    top_borough_codes.sort_values(['borough', 'tickets']),
    x='tickets', y='description', color='borough', orientation='h', facet_col='borough', facet_col_wrap=2,
    title='Top five violation types within each borough'
)
fig.update_layout(showlegend=False, yaxis_title='', xaxis_title='Tickets')
fig

### Result: Boroughs Have Different Ticket Profiles

School-zone speed tickets dominate Brooklyn, Queens, Bronx, and Staten Island. Manhattan is different: meter receipt, no-standing, street-cleaning, and commercial-meter-zone violations are more central. This means a single citywide top-ten chart hides important borough-level structure.

In [ ]:
# What is inside precinct 0?
precinct_zero_codes.head(12)

In [ ]:
fig = px.bar(
    precinct_zero_codes.head(10).sort_values('tickets'),
    x='tickets', y='description', orientation='h',
    title='Top violations recorded with precinct 0'
)
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

In [ ]:
# Compare precinct-zero records with mappable non-zero precinct records.
nonzero_precinct_codes.head(12)

In [ ]:
fig = px.bar(
    nonzero_precinct_codes.head(10).sort_values('tickets'),
    x='tickets', y='description', orientation='h',
    title='Top violations recorded with non-zero precincts'
)
fig.update_layout(yaxis_title='', xaxis_title='Tickets')
fig

### Result: Precinct `0` Should Not Be Mapped As A Precinct

Precinct `0` is dominated by school-zone speed, bus-lane, red-light, and MTA camera violations. Non-zero precinct records look like traditional curbside parking enforcement: street cleaning, meter receipt, no-standing, hydrant, inspection sticker, registration sticker, and double-parking violations. For maps, precinct `0` should be reported separately, not forced into a precinct polygon.

In [ ]:
# Borough dependence on camera-like enforcement.
channel_plot = channel_by_borough[channel_by_borough['borough'].isin(BOROUGHS)].sort_values('camera_share')
fig = px.bar(channel_plot, x='camera_share', y='borough', orientation='h',
             title='Share of borough tickets from camera-like enforcement')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='Camera-like share', yaxis_title='')
fig

### Result: The Geographic Story Changes When Cameras Are Included

Staten Island and Queens have much higher camera-like shares than Manhattan. Manhattan has the largest curbside-ticket count, while Queens and Brooklyn look much larger when camera-like tickets are included. The final story should therefore separate **automated enforcement geography** from **curbside enforcement geography**.

## New Recommendation: Where To Go Next

The strongest next direction is a two-track story:

1. **Automated enforcement city**: school-zone speed, red-light, bus-lane, and MTA camera records. Show how these dominate precinct `0` and shift the borough ranking toward Queens, Brooklyn, Bronx, and Staten Island.
2. **Curbside enforcement city**: street cleaning, meters, no-standing, hydrants, stickers, and double parking. Map these by borough now, then by real precinct after joining a precinct GeoJSON.

For the final project, the most defensible narrative is: **NYC's parking-violation geography depends on what kind of enforcement you count. Camera-like enforcement and curbside enforcement produce different maps, different top violations, and different policy questions.**

Concrete next tasks:

- Build a precinct choropleth using only non-zero precinct records.
- Add a dropdown or small multiples comparing camera-like vs curbside borough maps.
- Use fine estimates to compare ticket volume with estimated fine value.
- Add one careful caveat section: raw counts are not fairness measures without exposure data such as population, cars, curb space, school-zone cameras, meters, or neighborhood income.

## Next Analysis Step: Precincts And Fine Value

Now that precinct `0` is separated, we can map real precincts and compare ticket volume with estimated fine value. This tests whether the biggest ticket categories are also the biggest money categories.

In [ ]:
precincts = pd.read_csv(EDA_DIR / 'precincts.csv', dtype={'precinct': str})
fine_rank = pd.read_csv(EDA_DIR / 'fine_rank_comparison.csv', dtype={'code': str})

precincts.head(), fine_rank.head()

In [ ]:
PRECINCT_GEOJSON_URL = 'https://data.cityofnewyork.us/resource/y76i-bdw7.geojson?$limit=5000'
PRECINCT_GEOJSON_PATH = OUTPUT_DIR / 'police_precincts.geojson'

if not PRECINCT_GEOJSON_PATH.exists():
    urllib.request.urlretrieve(PRECINCT_GEOJSON_URL, PRECINCT_GEOJSON_PATH)

with open(PRECINCT_GEOJSON_PATH) as f:
    precinct_geojson = json.load(f)

geo_precincts = {feature['properties']['precinct'] for feature in precinct_geojson['features']}
real_precincts = precincts[precincts['precinct'].ne('0')].copy()
map_precincts = real_precincts[real_precincts['precinct'].isin(geo_precincts)].copy()
missing_precincts = sorted(set(real_precincts['precinct']) - geo_precincts)

print('Mappable precincts:', len(map_precincts))
print('Non-mappable non-zero precinct labels:', len(missing_precincts))
print(missing_precincts[:20])

In [ ]:
fig = px.choropleth_map(
    map_precincts,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='properties.precinct',
    color='tickets',
    hover_name='precinct',
    hover_data={'tickets': ':,'},
    color_continuous_scale='Viridis',
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.82,
    title='FY2025 parking violations by real NYPD precinct, excluding precinct 0'
)
fig.update_traces(marker_line_width=0.7, marker_line_color='white')
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.write_html(OUTPUT_DIR / 'eda_precinct_nonzero_map.html')
fig

### Precinct Map Result

The precinct choropleth is now valid because it excludes precinct `0`. It also reveals a data-quality issue: many non-zero labels in the ticket file do not match current NYPD precinct polygons. Those labels should be treated as administrative/location codes unless verified, not blindly mapped.

In [ ]:
fig = px.scatter(
    fine_rank.head(20),
    x='tickets', y='estimated_fines_other_areas', text='code',
    hover_name='violation_description_dict',
    title='Ticket volume vs estimated fine value, top 20 fine-value codes',
    labels={'tickets': 'Tickets', 'estimated_fines_other_areas': 'Estimated fine value ($)'}
)
fig.update_traces(textposition='top center')
fig

In [ ]:
# Categories whose money rank is much higher than their ticket-count rank.
fine_rank.sort_values('rank_shift', ascending=False).head(10)[[
    'code', 'violation_description_dict', 'tickets', 'fine_other_areas',
    'estimated_fines_other_areas', 'ticket_rank', 'value_rank', 'rank_shift'
]]

### Fine-Value Result

School-zone speed is first by both count and estimated fine value. But high-fine categories such as no-standing, fire hydrant, double parking, and commercial-meter-zone violations move up when we rank by estimated dollars instead of ticket counts. This gives a stronger financial angle than a simple top-ticket chart.

## Updated Recommendation

The project should now move toward a three-part final story:

1. **Scale**: NYC issued more than 16 million FY2025 records in this file, and school-zone speed alone is the largest category.
2. **Two enforcement geographies**: camera-like enforcement dominates precinct `0` and changes borough rankings; curbside enforcement maps differently and can be shown by real precinct.
3. **Money vs volume**: the most common violations are not always the categories that rise most in estimated fine value.

Best next step: build a final narrative with three visuals:

- A static bar chart comparing top violations by ticket count and estimated fine value.
- A borough or precinct choropleth for curbside enforcement only.
- An interactive view or small multiples comparing camera-like vs curbside enforcement by borough.

Avoid making an equity claim yet. The data supports an enforcement-geography story, but fairness analysis needs denominators such as population, car ownership, curb space, meters, camera locations, commute flow, and neighborhood income.

## Story Candidate Visuals

The next step is to convert the exploratory results into three reusable story figures: volume vs fine value, borough enforcement channel, and camera-share concentration.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

STORY_DIR = OUTPUT_DIR / 'story'
STORY_DIR.mkdir(exist_ok=True)

fine_rank = pd.read_csv(EDA_DIR / 'fine_rank_comparison.csv', dtype={'code': str})
channel_by_borough = pd.read_csv(EDA_DIR / 'channel_by_borough.csv')
channel_by_borough = channel_by_borough[channel_by_borough['borough'].isin(BOROUGHS)].copy()
channel_by_borough['total'] = channel_by_borough['camera_like'] + channel_by_borough['curbside']

In [ ]:
top = fine_rank.head(12).iloc[::-1]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=('Tickets', 'Estimated fine value'))
fig.add_trace(go.Bar(x=top['tickets'], y=top['violation_description_dict'], orientation='h', marker_color='#247BA0'), row=1, col=1)
fig.add_trace(go.Bar(x=top['estimated_fines_other_areas'], y=top['violation_description_dict'], orientation='h', marker_color='#F25F5C'), row=1, col=2)
fig.update_layout(title='Top parking violations by volume and estimated fine value', height=620, showlegend=False)
fig.write_html(STORY_DIR / 'fig1_count_vs_fine_value.html')
fig

In [ ]:
channel_long = channel_by_borough.melt(
    id_vars='borough', value_vars=['curbside', 'camera_like'],
    var_name='channel', value_name='tickets'
)
channel_long['channel'] = channel_long['channel'].map({'curbside': 'Curbside', 'camera_like': 'Camera-like'})

fig = px.bar(channel_long, x='borough', y='tickets', color='channel', barmode='stack',
             title='Camera-like and curbside tickets by borough',
             color_discrete_map={'Curbside': '#2A9D8F', 'Camera-like': '#E9C46A'})
fig.update_layout(xaxis_title='', yaxis_title='Tickets', legend_title='')
fig.write_html(STORY_DIR / 'fig2_borough_channel_split.html')
fig

In [ ]:
share = channel_by_borough.sort_values('camera_share')
fig = px.bar(share, x='camera_share', y='borough', orientation='h', color='camera_share',
             color_continuous_scale='Viridis', title='Camera-like share of tickets by borough')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='Camera-like share', yaxis_title='', coloraxis_showscale=False)
fig.write_html(STORY_DIR / 'fig3_borough_camera_share.html')
fig

In [ ]:
story_metrics = {
    'fy_rows': int(quality.loc[0, 'fy_rows']),
    'precinct_zero_share': float(quality.loc[0, 'precinct_zero'] / quality.loc[0, 'fy_rows']),
    'top_fine_description': fine_rank.iloc[0]['violation_description_dict'],
    'top_fine_estimate': float(fine_rank.iloc[0]['estimated_fines_other_areas']),
    'highest_camera_share_borough': channel_by_borough.sort_values('camera_share', ascending=False).iloc[0]['borough'],
    'largest_curbside_borough': channel_by_borough.sort_values('curbside', ascending=False).iloc[0]['borough'],
}
story_metrics

## New Suggestion

The best direction is now a focused narrative, not more broad EDA. The evidence supports this claim:

**NYC parking enforcement is really two systems layered together: automated/camera enforcement and curbside enforcement. They produce different geographies and different money stories.**

Recommended final structure:

1. **Open with scale and money**: show that school-zone speed dominates both ticket count and estimated fine value.
2. **Separate the two systems**: use borough bars to show where camera-like enforcement changes the story.
3. **Map only what is mappable**: use the non-zero precinct map for curbside/real precinct enforcement, while explicitly keeping precinct `0` separate.

Next concrete task: turn these three figures into a short magazine-style data story page with captions, caveats, and exported HTML assets.

## Next Analysis Step: Timing By Enforcement Channel

The draft asks *when* tickets happen. We now compare timing for `Precinct 0` records versus real-precinct records. This is a cleaner split than the earlier camera-code shortcut because it uses the administrative field that caused the mapping problem.

In [ ]:
channel_hour = pd.read_csv(EDA_DIR / 'channel_hour.csv')
channel_weekday = pd.read_csv(EDA_DIR / 'channel_weekday.csv')
channel_month = pd.read_csv(EDA_DIR / 'channel_month.csv')
channel_hour_weekday = pd.read_csv(EDA_DIR / 'channel_hour_weekday.csv')

channel_hour.head()

In [ ]:
channel_hour['channel_total'] = channel_hour.groupby('channel')['tickets'].transform('sum')
channel_hour['share'] = channel_hour['tickets'] / channel_hour['channel_total']

fig = px.line(channel_hour, x='hour', y='share', color='channel', markers=True,
              title='Hourly ticket profile by enforcement channel',
              labels={'hour': 'Hour of day', 'share': 'Share of channel tickets', 'channel': ''})
fig.update_layout(yaxis_tickformat='.1%')
fig.write_html(STORY_DIR / 'fig4_channel_hour_profile.html')
fig

In [ ]:
channel_weekday['weekday'] = pd.Categorical(channel_weekday['weekday'], categories=WEEKDAYS, ordered=True)
channel_weekday = channel_weekday.sort_values(['channel', 'weekday'])
channel_weekday['channel_total'] = channel_weekday.groupby('channel')['tickets'].transform('sum')
channel_weekday['share'] = channel_weekday['tickets'] / channel_weekday['channel_total']

fig = px.bar(channel_weekday, x='weekday', y='share', color='channel', barmode='group',
             title='Weekday ticket profile by enforcement channel',
             labels={'weekday': '', 'share': 'Share of channel tickets', 'channel': ''})
fig.update_layout(yaxis_tickformat='.1%')
fig.write_html(STORY_DIR / 'fig5_channel_weekday_profile.html')
fig

In [ ]:
month_split = channel_month.pivot(index='month', columns='channel', values='tickets').fillna(0).reset_index()
month_split['precinct0_share'] = month_split['Precinct 0'] / (month_split['Precinct 0'] + month_split['Real precinct'])

fig = px.line(month_split, x='month', y='precinct0_share', markers=True,
              title='Precinct 0 share by month', labels={'month': '', 'precinct0_share': 'Precinct 0 share'})
fig.update_layout(yaxis_tickformat='.0%')
fig.write_html(STORY_DIR / 'fig6_monthly_precinct0_share.html')
fig

### Timing Result

`Precinct 0` records peak in the afternoon, while real-precinct records peak in the morning. This supports the two-system interpretation: camera/admin enforcement and curbside enforcement do not only map differently; they also operate on different daily rhythms.

In [ ]:
timing_summary = pd.DataFrame([
    {
        'channel': channel,
        'peak_hour': int(channel_hour[channel_hour['channel'].eq(channel)].sort_values('tickets', ascending=False).iloc[0]['hour']),
        'peak_weekday': channel_weekday[channel_weekday['channel'].eq(channel)].sort_values('tickets', ascending=False).iloc[0]['weekday'],
    }
    for channel in ['Precinct 0', 'Real precinct']
])
timing_summary

## Updated Suggestion After Timing Analysis

The final story is stronger if it uses **time** as the third dimension, not only money. The clearest evidence chain is now:

1. **Scale**: FY2025 has more than 16.2 million valid rows, and almost half are `Precinct 0`.
2. **Geography**: `Precinct 0` cannot be mapped as a normal precinct; real-precinct records can be mapped separately.
3. **Timing**: `Precinct 0` peaks in the afternoon, while real-precinct enforcement peaks in the morning.

Recommended next move: build the final narrative around **two enforcement systems** using three visuals: count/fine overview, borough or precinct map, and timing profile. Keep the fine-value chart as supporting evidence, but make the main story about how automated/admin enforcement and curbside enforcement create different patterns.

## Next Analysis Step: Violation Families

Individual violation codes are too detailed for the final story, so this step groups them into readable families: school-zone speed, bus lane/bus stop, street cleaning, no-standing/no-parking, meters, stickers, hydrants, red light, double parking, and other.

In [ ]:
family_totals = pd.read_csv(EDA_DIR / 'family_totals.csv')
family_channel_summary = pd.read_csv(EDA_DIR / 'family_channel_summary.csv')
family_borough = pd.read_csv(EDA_DIR / 'family_borough.csv')
family_hour = pd.read_csv(EDA_DIR / 'family_hour.csv')

family_totals

In [ ]:
fig = px.bar(family_totals.sort_values('tickets'), x='tickets', y='family', orientation='h',
             title='Violation families by ticket volume')
fig.update_layout(xaxis_title='Tickets', yaxis_title='')
fig.write_html(STORY_DIR / 'fig8_violation_family_totals.html')
fig

In [ ]:
fig = px.bar(family_channel_summary.sort_values('precinct0_share'), x='precinct0_share', y='family',
             orientation='h', color='precinct0_share', color_continuous_scale='Viridis',
             title='How much of each violation family is precinct 0?')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='Precinct 0 share', yaxis_title='', coloraxis_showscale=False)
fig.write_html(STORY_DIR / 'fig9_family_precinct0_share.html')
fig

In [ ]:
top_families = family_totals.head(8)['family'].tolist()
family_borough_plot = family_borough[
    family_borough['borough'].isin(BOROUGHS) & family_borough['family'].isin(top_families)
]

fig = px.bar(family_borough_plot, x='borough', y='tickets', color='family', barmode='stack',
             title='Top violation families by borough')
fig.update_layout(xaxis_title='', yaxis_title='Tickets', legend_title='')
fig.write_html(STORY_DIR / 'fig10_family_by_borough.html')
fig

In [ ]:
family_hour_plot = family_hour[family_hour['family'].isin(top_families)].copy()
family_hour_plot['family_total'] = family_hour_plot.groupby('family')['tickets'].transform('sum')
family_hour_plot['share'] = family_hour_plot['tickets'] / family_hour_plot['family_total']

fig = px.line(family_hour_plot, x='hour', y='share', color='family',
              title='Hourly profiles of major violation families')
fig.update_layout(yaxis_tickformat='.1%', xaxis_title='Hour of day', yaxis_title='Share within family', legend_title='')
fig.write_html(STORY_DIR / 'fig11_family_hour_profiles.html')
fig

### Family Result

The two-system story is still valid, but now it is more specific. `School-zone speed` is the largest family and is entirely precinct `0`. `Meter / paid parking` is almost entirely real-precinct enforcement. `Bus lane / bus stop`, `red light`, and `MTA camera double parking` explain much of the remaining automated/admin side. The final story should name these families rather than relying only on abstract labels like “camera-like.”

## Updated Suggestion After Family Analysis

The final project should now use **violation families** as the reader-facing language.

Recommended final claim:

**NYC’s FY2025 parking-ticket dataset combines several enforcement systems. School-zone speed and bus-camera records dominate the automated side, while meters, street cleaning, no-standing/no-parking, hydrants, and sticker violations define curbside enforcement. These systems differ by geography, time of day, and estimated fine value.**

Best next step: stop adding new analyses and start assembling the final narrative page. Use three or four visuals:

1. Violation-family volume chart.
2. Borough channel split or family-by-borough chart.
3. Timing profile by channel or family.
4. Optional fine-value chart as the money angle.

This is now a coherent story with enough evidence for the final project.

## Next Analysis Step: Precinct Specialization

We already mapped total tickets by precinct. This step asks a different question: **what type of enforcement defines each mapped precinct?** It uses only non-zero precincts that match the NYPD precinct GeoJSON.

In [ ]:
family_precinct = pd.read_csv(EDA_DIR / 'family_precinct.csv', dtype={'precinct': str})
dominant_family = pd.read_csv(EDA_DIR / 'dominant_family_by_precinct.csv', dtype={'precinct': str})
family_specialization = pd.read_csv(EDA_DIR / 'family_precinct_specialization.csv', dtype={'precinct': str})

family_precinct.head()

In [ ]:
fig = px.choropleth_map(
    dominant_family,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='properties.precinct',
    color='family',
    hover_name='precinct',
    hover_data={'tickets': ':,', 'family_share_in_precinct': ':.1%', 'precinct_total': ':,'},
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.85,
    title='Dominant violation family by mapped NYPD precinct'
)
fig.update_traces(marker_line_width=0.7, marker_line_color='white')
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0}, legend_title='Dominant family')
fig.write_html(STORY_DIR / 'fig12_dominant_family_by_precinct.html')
fig

In [ ]:
street_cleaning = family_precinct[family_precinct['family'].eq('Street cleaning')]
fig = px.choropleth_map(
    street_cleaning,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='properties.precinct',
    color='family_share_in_precinct',
    hover_name='precinct',
    hover_data={'tickets': ':,', 'family_share_in_precinct': ':.1%', 'specialization_ratio': ':.2f'},
    color_continuous_scale='Viridis',
    map_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.85,
    title='Street-cleaning share within each mapped precinct'
)
fig.update_traces(marker_line_width=0.7, marker_line_color='white')
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0}, coloraxis_colorbar={'tickformat': '.0%'})
fig.write_html(STORY_DIR / 'fig13_street_cleaning_precinct_share.html')
fig

In [ ]:
fig = px.bar(
    family_specialization.head(15).sort_values('specialization_ratio'),
    x='specialization_ratio', y='precinct', color='family', orientation='h',
    title='Strongest precinct specializations by violation family',
    hover_data={'tickets': ':,', 'family_share_in_precinct': ':.1%'}
)
fig.update_layout(xaxis_title='Specialization ratio vs mapped-precinct average', yaxis_title='Precinct', legend_title='Family')
fig.write_html(STORY_DIR / 'fig14_precinct_family_specialization.html')
fig

### Precinct-Specialization Result

Among 78 mapped precincts, `Street cleaning` is the dominant family in 44. That makes it the clearest real-precinct curbside geography. The strongest specialization is precinct `122`, where `Registration / inspection sticker` tickets are about 3.68 times more prominent than the mapped-precinct average. This is a more precise spatial finding than raw precinct totals.

## Updated Suggestion After Precinct Specialization

The story can now move from “two enforcement systems” to a sharper final angle:

**Automated enforcement explains precinct `0`, but real precinct geography is mostly a curb-management story, led by street cleaning and localized specializations such as stickers, meters, double parking, and hydrants.**

Best next step: build the final narrative page. Use:

1. `fig8_violation_family_totals.html` to introduce enforcement families.
2. `fig12_dominant_family_by_precinct.html` or `fig13_street_cleaning_precinct_share.html` for the spatial argument.
3. `fig4_channel_hour_profile.html` or `fig11_family_hour_profiles.html` for the timing argument.
4. `fig1_count_vs_fine_value.html` as the money angle if the story needs one more figure.

Further EDA is now likely to dilute the story unless we add external denominator data.

## Next Analysis Step: Family-Level Fine Value

The earlier fine chart worked at the individual-code level. Now that the story uses violation families, this step estimates fine value by family. This avoids mixing a code-level money chart with a family-level narrative.

In [ ]:
family_fine = pd.read_csv(EDA_DIR / 'family_fine_estimates.csv')
family_fine[[
    'family', 'tickets', 'estimated_fines_other_areas',
    'avg_estimated_fine_other_areas', 'ticket_rank', 'value_rank', 'rank_shift'
]]

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

plot = family_fine.sort_values('estimated_fines_other_areas')
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=('Tickets', 'Estimated fine value'))
fig.add_trace(go.Bar(x=plot['tickets'], y=plot['family'], orientation='h', marker_color='#247BA0'), row=1, col=1)
fig.add_trace(go.Bar(x=plot['estimated_fines_other_areas'], y=plot['family'], orientation='h', marker_color='#F25F5C'), row=1, col=2)
fig.update_layout(title='Violation families by ticket volume and estimated fine value', height=600, showlegend=False)
fig.write_html(STORY_DIR / 'fig15_family_count_vs_fine_value.html')
fig

In [ ]:
avg_fine = family_fine.dropna(subset=['avg_estimated_fine_other_areas']).sort_values('avg_estimated_fine_other_areas')
fig = px.bar(
    avg_fine,
    x='avg_estimated_fine_other_areas', y='family', orientation='h',
    title='Average estimated fine per ticket by violation family',
    hover_data={'tickets': ':,', 'known_fine_ticket_share': ':.1%'}
)
fig.update_layout(xaxis_title='Average estimated fine ($)', yaxis_title='')
fig.write_html(STORY_DIR / 'fig16_family_average_fine.html')
fig

### Family Fine Result

School-zone speed remains the largest family by both ticket count and estimated fine value. But the money ranking changes after that: `No standing / no parking`, `Hydrant`, `Double parking`, and `MTA camera double parking` become more important because their estimated per-ticket fines are high. `Meter / paid parking` is common but falls in the fine-value ranking because its estimated fine is comparatively low.

## Updated Suggestion After Family Fine Analysis

The final story should not present “money” as a separate code-level appendix. It should be integrated into the family story:

**Some enforcement families are big because they produce many tickets; others matter because each ticket is expensive.**

Best next move: create the final narrative page. Use the family-level money chart instead of the older code-level money chart, because it matches the story language. The strongest final visual set is now:

1. `fig8_violation_family_totals.html` for scale.
2. `fig12_dominant_family_by_precinct.html` or `fig13_street_cleaning_precinct_share.html` for geography.
3. `fig11_family_hour_profiles.html` for timing.
4. `fig15_family_count_vs_fine_value.html` for money.

Further analysis should only happen if the story draft exposes a missing claim.

## Missing Angle: Vehicle And Plate Profile

So far, the analysis has focused on tickets, places, times, violation families, and estimated fines. A different unused angle is the vehicle side: registration state, plate type, and vehicle attributes. This asks whether the dataset is mostly local passenger vehicles or whether out-of-state, commercial, taxi/livery, and rental plates tell a different story.

In [ ]:
vehicle_state_groups = pd.read_csv(EDA_DIR / 'vehicle_state_groups.csv')
vehicle_plate_classes = pd.read_csv(EDA_DIR / 'vehicle_plate_classes.csv')
vehicle_state_family = pd.read_csv(EDA_DIR / 'vehicle_state_family.csv')
vehicle_plate_family = pd.read_csv(EDA_DIR / 'vehicle_plate_family.csv')
vehicle_state_channel = pd.read_csv(EDA_DIR / 'vehicle_state_channel.csv')
vehicle_plate_channel = pd.read_csv(EDA_DIR / 'vehicle_plate_channel.csv')
vehicle_year_quality = pd.read_csv(EDA_DIR / 'vehicle_year_quality.csv')

vehicle_state_groups, vehicle_plate_classes, vehicle_year_quality

In [ ]:
state_family_mix = vehicle_state_family.merge(vehicle_state_groups, on='state_group', suffixes=('', '_state_total'))
state_family_mix['share'] = state_family_mix['tickets'] / state_family_mix['tickets_state_total']
family_keep = ['School-zone speed', 'Bus lane / bus stop', 'Street cleaning', 'No standing / no parking',
               'Meter / paid parking', 'Registration / inspection sticker', 'Hydrant', 'Red light']

fig = px.bar(
    state_family_mix[state_family_mix['family'].isin(family_keep)],
    x='state_group', y='share', color='family', barmode='stack',
    title='Violation-family mix by registration state group'
)
fig.update_layout(xaxis_title='', yaxis_title='Share within state group', yaxis_tickformat='.0%', legend_title='')
fig.write_html(STORY_DIR / 'fig17_state_group_family_mix.html')
fig

In [ ]:
plate_family_mix = vehicle_plate_family.merge(vehicle_plate_classes, on='plate_class', suffixes=('', '_plate_total'))
plate_family_mix['share'] = plate_family_mix['tickets'] / plate_family_mix['tickets_plate_total']
major_plate_classes = vehicle_plate_classes.head(5)['plate_class'].tolist()

fig = px.bar(
    plate_family_mix[plate_family_mix['plate_class'].isin(major_plate_classes) & plate_family_mix['family'].isin(family_keep)],
    x='plate_class', y='share', color='family', barmode='stack',
    title='Violation-family mix by plate class'
)
fig.update_layout(xaxis_title='', yaxis_title='Share within plate class', yaxis_tickformat='.0%', legend_title='')
fig.write_html(STORY_DIR / 'fig18_plate_class_family_mix.html')
fig

In [ ]:
state_channel = vehicle_state_channel.pivot(index='state_group', columns='channel', values='tickets').fillna(0).reset_index()
state_channel['total'] = state_channel.get('Precinct 0', 0) + state_channel.get('Real precinct', 0)
state_channel['precinct0_share'] = state_channel.get('Precinct 0', 0) / state_channel['total']

fig = px.bar(state_channel.sort_values('precinct0_share'), x='precinct0_share', y='state_group', orientation='h',
             title='Precinct 0 share by registration state group')
fig.update_layout(xaxis_title='Precinct 0 share', xaxis_tickformat='.0%', yaxis_title='')
fig.write_html(STORY_DIR / 'fig19_state_group_precinct0_share.html')
fig

In [ ]:
plate_channel = vehicle_plate_channel.pivot(index='plate_class', columns='channel', values='tickets').fillna(0).reset_index()
plate_channel['total'] = plate_channel.get('Precinct 0', 0) + plate_channel.get('Real precinct', 0)
plate_channel['precinct0_share'] = plate_channel.get('Precinct 0', 0) / plate_channel['total']
plate_channel = plate_channel[plate_channel['total'].ge(10_000)]

fig = px.bar(plate_channel.sort_values('precinct0_share'), x='precinct0_share', y='plate_class', orientation='h',
             title='Precinct 0 share by plate class')
fig.update_layout(xaxis_title='Precinct 0 share', xaxis_tickformat='.0%', yaxis_title='')
fig.write_html(STORY_DIR / 'fig20_plate_class_precinct0_share.html')
fig

### Vehicle-Profile Result

The dataset is mostly NY-registered passenger/personal vehicles, but the non-local and non-passenger groups are large enough to matter. About 71.5% of FY2025 tickets are NY-registered, 16.6% are from neighboring states, 83.7% are passenger/personal plates, 9.3% are commercial/carrier plates, and 5.0% are taxi/livery/rental plates. Vehicle year is usable for about 82.7% of rows, so it can support a later vehicle-age check if needed.

## Updated Suggestion After Vehicle Analysis

This vehicle angle is useful, but it should probably be a **secondary section**, not the main story. The main story is already strong: enforcement systems differ by geography, timing, and violation family. Vehicle profile adds a “who/what vehicles show up in the records?” layer.

Best use in the final project:

- Add one short section after the main enforcement-system story: **Most tickets are NY passenger vehicles, but neighboring-state and commercial/taxi/rental plates form meaningful subgroups.**
- Use `fig18_plate_class_family_mix.html` if you want a vehicle-focused visual.
- Do not pivot the whole project to vehicles unless you decide to add a deeper question about commuters, commercial activity, or taxi/rental enforcement.

The next useful task is no longer more analysis. It is editorial: choose the final 3-4 figures and write the narrative around them.

## Missing Angle: Street And Curb Metadata

Another underused part of the dataset is the curb/location metadata: street name, house number, intersecting street, meter number, violation post code, and posted rule hours. This tests whether we can tell a street-corridor story, not just borough or precinct stories.

In [ ]:
street_totals = pd.read_csv(EDA_DIR / 'street_totals.csv')
borough_street = pd.read_csv(EDA_DIR / 'borough_street.csv')
street_ambiguity = pd.read_csv(EDA_DIR / 'street_borough_ambiguity.csv')
location_quality = pd.read_csv(EDA_DIR / 'location_quality.csv')
location_quality_by_family = pd.read_csv(EDA_DIR / 'location_quality_by_family.csv')

street_totals.head()

In [ ]:
street_rank = street_totals.sort_values('tickets', ascending=False).copy()
street_rank['rank'] = range(1, len(street_rank) + 1)
street_rank['cum_share'] = street_rank['tickets'].cumsum() / street_rank['tickets'].sum()

fig = px.line(street_rank.head(1000), x='rank', y='cum_share',
              title='Street-name concentration: cumulative share of tickets')
fig.update_layout(yaxis_tickformat='.0%', xaxis_title='Street-name rank', yaxis_title='Cumulative ticket share')
fig.write_html(STORY_DIR / 'fig21_street_concentration_curve.html')
fig

In [ ]:
fig = px.bar(street_rank.head(20).sort_values('tickets'), x='tickets', y='street', orientation='h',
             title='Top street-name labels by tickets')
fig.update_layout(xaxis_title='Tickets', yaxis_title='')
fig.write_html(STORY_DIR / 'fig22_top_street_names.html')
fig

In [ ]:
top_borough_street = (
    borough_street[borough_street['borough'].isin(BOROUGHS)]
    .sort_values(['borough', 'tickets'], ascending=[True, False])
    .groupby('borough')
    .head(8)
)

fig = px.bar(top_borough_street.sort_values(['borough', 'tickets']), x='tickets', y='street',
             color='borough', orientation='h', facet_col='borough', facet_col_wrap=2,
             title='Top street labels within each borough')
fig.update_layout(showlegend=False, xaxis_title='Tickets', yaxis_title='')
fig.write_html(STORY_DIR / 'fig23_top_streets_by_borough.html')
fig

In [ ]:
field_cols = [
    'has_house_share', 'has_intersection_share', 'has_meter_share', 'has_post_code_share',
    'has_from_hours_share', 'has_to_hours_share', 'has_days_effect_share'
]
loc_family_long = location_quality_by_family[['family'] + field_cols].melt(
    id_vars='family', var_name='field', value_name='share'
)
loc_family_long['field'] = (
    loc_family_long['field']
    .str.replace('has_', '', regex=False)
    .str.replace('_share', '', regex=False)
    .str.replace('_', ' ')
)

fig = px.bar(loc_family_long, x='family', y='share', color='field', barmode='group',
             title='Location metadata completeness by violation family')
fig.update_layout(xaxis_title='', yaxis_title='Share of rows', yaxis_tickformat='.0%', legend_title='Field')
fig.write_html(STORY_DIR / 'fig24_location_completeness_by_family.html')
fig

### Street / Curb Metadata Result

Street names are highly concentrated: the top 100 street-name labels account for about 22.2% of tickets, and `BROADWAY` alone has 179,263 records. But street-level analysis is risky because 9,103 street labels appear in more than one borough. House number is present in about 53.0% of rows, intersection in 54.0%, meter number in 12.2%, and violation post code in 35.0%. This supports a careful corridor/concentration angle, but not precise block-level mapping without additional geocoding or cleaning.

## Updated Suggestion After Street Analysis

This is a valid new angle, but it should be used as a **caveat or supporting section**, not the main final story.

Possible addition to the final narrative:

**Tickets are concentrated on a relatively small set of street-name corridors, but the location fields are too inconsistent for exact block-level claims.**

Use this if you want a methodological honesty section: it explains why we chose borough/precinct/family maps instead of exact street hot-spot maps. The main story should still remain enforcement systems, violation families, geography, timing, and fine value.

## Missing Angle: Repeat Vehicle Keys

Another unused angle is ticket concentration across vehicle keys. To avoid exposing plate IDs, this analysis uses only anonymized and bucketed summaries of `(registration state, plate type, plate id)` keys. Treat these as plate records, not proven people or unique vehicles, because plates can be reused, mistyped, or fleet-managed.

In [ ]:
repeat_distribution = pd.read_csv(EDA_DIR / 'repeat_vehicle_distribution.csv')
repeat_state = pd.read_csv(EDA_DIR / 'repeat_vehicle_by_state_group.csv')
repeat_plate = pd.read_csv(EDA_DIR / 'repeat_vehicle_by_plate_class.csv')
top_repeat_anonymous = pd.read_csv(EDA_DIR / 'top_repeat_vehicle_anonymous.csv')

repeat_distribution

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

bucket_order = ['1', '2', '3-5', '6-10', '11-25', '26-50', '51+']
repeat_distribution['ticket_bucket'] = pd.Categorical(repeat_distribution['ticket_bucket'], categories=bucket_order, ordered=True)
repeat_distribution = repeat_distribution.sort_values('ticket_bucket')

fig = make_subplots(rows=1, cols=2, subplot_titles=('Share of vehicle keys', 'Share of tickets'))
fig.add_trace(go.Bar(x=repeat_distribution['ticket_bucket'], y=repeat_distribution['vehicle_share'], marker_color='#2A9D8F'), row=1, col=1)
fig.add_trace(go.Bar(x=repeat_distribution['ticket_bucket'], y=repeat_distribution['ticket_share'], marker_color='#E76F51'), row=1, col=2)
fig.update_layout(title='Repeat vehicle concentration by ticket bucket', showlegend=False)
fig.update_yaxes(tickformat='.0%')
fig.write_html(STORY_DIR / 'fig25_repeat_vehicle_distribution.html')
fig

In [ ]:
fig = px.bar(repeat_state.sort_values('tickets_per_vehicle'), x='tickets_per_vehicle', y='state_group', orientation='h',
             title='Tickets per vehicle key by registration state group')
fig.update_layout(xaxis_title='Tickets per vehicle key', yaxis_title='')
fig.write_html(STORY_DIR / 'fig26_tickets_per_vehicle_state_group.html')
fig

In [ ]:
repeat_plate_plot = repeat_plate[repeat_plate['vehicles'].ge(1000)].sort_values('tickets_per_vehicle')
fig = px.bar(repeat_plate_plot, x='tickets_per_vehicle', y='plate_class', orientation='h', color='repeat_vehicle_share',
             color_continuous_scale='Viridis', title='Tickets per vehicle key by plate class')
fig.update_layout(xaxis_title='Tickets per vehicle key', yaxis_title='', coloraxis_colorbar={'title': 'Repeat share', 'tickformat': '.0%'})
fig.write_html(STORY_DIR / 'fig27_tickets_per_vehicle_plate_class.html')
fig

In [ ]:
fig = px.line(top_repeat_anonymous.head(50), x='anonymous_rank', y='tickets', color='plate_class', markers=True,
              title='Top repeat vehicle keys, anonymized')
fig.update_layout(xaxis_title='Anonymous rank', yaxis_title='Tickets')
fig.write_html(STORY_DIR / 'fig28_top_repeat_vehicle_anonymous.html')
fig

### Repeat-Vehicle Result

The dataset is highly concentrated by vehicle key. About 43.4% of valid vehicle keys appear only once, but those one-time keys account for only 10.8% of tickets. Repeat keys account for about 89.2% of tickets. The top 1% of vehicle keys account for about 14.9% of tickets, and the highest anonymous key has 1,088 tickets. This suggests fleets, repeated exposure, camera-heavy routes, or unresolved repeat enforcement patterns may be important.

## Updated Suggestion After Repeat-Vehicle Analysis

This is the first new angle that could compete with the current final story. It reframes the project from “where and when tickets happen” to **how concentrated ticketing is across vehicle records**.

Possible final angle:

**NYC’s parking-ticket system is not only geographically concentrated; it is also concentrated across repeat vehicle keys. Most ticket records come from plates that appear more than once, while a small tail of vehicle keys accounts for a disproportionate share of tickets.**

However, this should be handled carefully because plate IDs are sensitive and imperfect identifiers. If used, only show bucketed/anonymized summaries. Do not display raw plate IDs.

Best next direction: decide between two final narratives:

1. **Enforcement-systems story**: automated/admin vs curbside enforcement, with geography/timing/fines.
2. **Concentration story**: tickets concentrate across violation families, streets, and repeat vehicle keys.

The strongest final project may combine them: automated and curbside systems create different patterns, and repeat vehicle keys reveal that ticket burden is also highly concentrated across records.

## Missing Angle: Observation-Based Enforcement

`Time First Observed` tells us whether an officer or agent recorded an earlier observation before issuing the ticket. This is a different enforcement mechanism from instant camera/admin records or simple timestamped curb tickets.

In [ ]:
observation_family = pd.read_csv(EDA_DIR / 'observation_by_family.csv')
observation_channel = pd.read_csv(EDA_DIR / 'observation_by_channel.csv')
observation_duration = pd.read_csv(EDA_DIR / 'observation_duration_buckets.csv')
observation_code = pd.read_csv(EDA_DIR / 'observation_by_code.csv', dtype={'code': str})

observation_family[['family', 'tickets', 'with_first_observed', 'first_observed_share', 'median_duration_min']]

In [ ]:
fig = px.bar(observation_family.sort_values('first_observed_share'), x='first_observed_share', y='family', orientation='h',
             title='Share of tickets with Time First Observed by violation family')
fig.update_layout(xaxis_title='Share with Time First Observed', xaxis_tickformat='.0%', yaxis_title='')
fig.write_html(STORY_DIR / 'fig29_first_observed_share_by_family.html')
fig

In [ ]:
obs_duration_family = observation_family.dropna(subset=['median_duration_min']).sort_values('median_duration_min')
fig = px.bar(obs_duration_family, x='median_duration_min', y='family', orientation='h',
             title='Median observation duration by violation family')
fig.update_layout(xaxis_title='Median minutes from first observed to violation time', yaxis_title='')
fig.write_html(STORY_DIR / 'fig30_observation_duration_by_family.html')
fig

In [ ]:
duration_order = ['0 min', '1-5 min', '6-15 min', '16-30 min', '31-60 min', '61-120 min', '120+ min']
observation_duration['duration_bucket'] = pd.Categorical(observation_duration['duration_bucket'], categories=duration_order, ordered=True)
observation_duration = observation_duration.sort_values('duration_bucket')

fig = px.bar(observation_duration, x='duration_bucket', y='tickets', title='Observation-duration buckets')
fig.update_layout(xaxis_title='Duration', yaxis_title='Tickets with valid duration')
fig.write_html(STORY_DIR / 'fig31_observation_duration_buckets.html')
fig

### Observation Result

`Time First Observed` is present in only about 5.8% of FY2025 rows, but it is not random. It is almost absent in `Precinct 0` records and appears in about 10.7% of real-precinct records. The strongest family is `Street cleaning`, where 28.6% of rows include a first-observed time. `Meter / paid parking` has a median valid observation duration of about 27 minutes. This suggests a small but meaningful “observed over time” enforcement layer inside the curbside system.

## Updated Suggestion After Observation Analysis

This is useful as a nuance, not the main storyline. It shows that curbside enforcement itself has subtypes:

- instant or near-instant records,
- camera/admin records,
- observed-over-time records such as meters and some street-cleaning tickets.

Best use: add one sentence or small supporting visual if explaining enforcement mechanisms. Do not make this the final project’s core unless you want a narrower technical story about how parking rules are recorded.

## Missing Angle: Issuing Agency And Enforcement Apparatus

Another unused part of the dataset is administrative: `Issuing Agency`, `Issuer Command`, `Issuer Squad`, `Issuer Code`, and `Issuer Precinct`. This asks which enforcement apparatus produces the records. Issuer codes are treated only in anonymized buckets.

In [ ]:
issuer_agency = pd.read_csv(EDA_DIR / 'issuer_agency_totals.csv')
issuer_agency_family = pd.read_csv(EDA_DIR / 'issuer_agency_family.csv')
issuer_agency_channel = pd.read_csv(EDA_DIR / 'issuer_agency_channel.csv')
issuer_code_buckets = pd.read_csv(EDA_DIR / 'issuer_code_bucket_distribution.csv')
issuer_precinct_match = pd.read_csv(EDA_DIR / 'issuer_precinct_match_by_agency.csv')
issuer_metrics = json.load(open(STORY_DIR / 'issuer_metrics.json'))

issuer_agency.head(10)

In [ ]:
fig = px.bar(issuer_agency.head(12).sort_values('tickets'), x='tickets', y='agency', orientation='h',
             title='Tickets by issuing agency code')
fig.update_layout(xaxis_title='Tickets', yaxis_title='Agency')
fig.write_html(STORY_DIR / 'fig33_issuing_agency_totals.html')
fig

In [ ]:
agency_channel_plot = issuer_agency_channel.pivot(index='agency', columns='channel', values='tickets').fillna(0).reset_index()
agency_channel_plot['total'] = agency_channel_plot.get('Precinct 0', 0) + agency_channel_plot.get('Real precinct', 0)
agency_channel_plot = agency_channel_plot[agency_channel_plot['total'].ge(10_000)].sort_values('total', ascending=False)
agency_channel_long = agency_channel_plot.melt(
    id_vars='agency', value_vars=[c for c in ['Real precinct', 'Precinct 0'] if c in agency_channel_plot.columns],
    var_name='channel', value_name='tickets'
)
fig = px.bar(agency_channel_long, x='agency', y='tickets', color='channel', barmode='stack',
             title='Issuing agency by enforcement channel')
fig.update_layout(xaxis_title='Agency', yaxis_title='Tickets', legend_title='')
fig.write_html(STORY_DIR / 'fig34_agency_channel_split.html')
fig

In [ ]:
top_agencies = issuer_agency.head(5)['agency'].tolist()
agency_family_plot = issuer_agency_family[issuer_agency_family['agency'].isin(top_agencies)].copy()
agency_family_plot['agency_total'] = agency_family_plot.groupby('agency')['tickets'].transform('sum')
agency_family_plot['share'] = agency_family_plot['tickets'] / agency_family_plot['agency_total']
family_keep = ['School-zone speed', 'Bus lane / bus stop', 'Street cleaning', 'No standing / no parking',
               'Meter / paid parking', 'Registration / inspection sticker', 'Hydrant', 'Red light', 'MTA camera double parking']
agency_family_plot = agency_family_plot[agency_family_plot['family'].isin(family_keep)]
fig = px.bar(agency_family_plot, x='agency', y='share', color='family', barmode='stack',
             title='Violation-family mix by top issuing agencies')
fig.update_layout(yaxis_tickformat='.0%', xaxis_title='Agency', yaxis_title='Share within agency', legend_title='')
fig.write_html(STORY_DIR / 'fig35_agency_family_mix.html')
fig

In [ ]:
bucket_order = ['1', '2', '3-5', '6-10', '11-25', '26-50', '51-100', '101-250', '251-500', '501-1000', '1001+']
issuer_code_buckets['ticket_bucket'] = pd.Categorical(issuer_code_buckets['ticket_bucket'], categories=bucket_order, ordered=True)
issuer_bucket_plot = issuer_code_buckets[issuer_code_buckets['agency'].isin(['T', 'S', 'P'])].sort_values(['agency', 'ticket_bucket'])
fig = px.bar(issuer_bucket_plot, x='ticket_bucket', y='tickets', color='agency', barmode='group',
             title='Anonymized issuer-code ticket buckets for selected agencies')
fig.update_layout(xaxis_title='Tickets per issuer-code bucket', yaxis_title='Tickets')
fig.write_html(STORY_DIR / 'fig36_issuer_code_bucket_distribution.html')
fig

In [ ]:
fig = px.bar(issuer_precinct_match[issuer_precinct_match['real_precinct_tickets'].ge(10_000)].sort_values('match_share'),
             x='match_share', y='agency', orientation='h',
             title='Issuer precinct matches violation precinct, by agency')
fig.update_layout(xaxis_title='Match share among real-precinct tickets', xaxis_tickformat='.0%', yaxis_title='Agency')
fig.write_html(STORY_DIR / 'fig37_issuer_precinct_match_share.html')
fig

### Issuer Result

Issuing agency `T` is the largest source of records and is overwhelmingly tied to real-precinct tickets. Agency `V` is dominated by precinct `0`, reinforcing the automated/admin enforcement channel. Valid issuer-code values exist for about 52.5% of rows, command for 53.8%, and squad for 50.1%. The top 1% of anonymized issuer codes account for about 27.9% of tickets with valid issuer codes, so ticket production is concentrated inside the administrative/enforcement apparatus too.

## Updated Suggestion After Issuer Analysis

This angle strengthens the main story because it gives institutional backing to the two-system split. The final narrative can now say:

**The split is visible not only in violations, maps, and timing, but also in the issuing apparatus: agency `T` aligns with real-precinct curbside records, while agency `V` aligns with precinct `0` automated/admin records.**

Use this sparingly. It is a strong supporting paragraph, but too administrative for the main headline. The main story should stay reader-facing: families, geography, timing, fine value, and concentration.

## Missing Angle: Registration And Expiration Status

The dataset includes `Vehicle Expiration Date` and `Unregistered Vehicle?`, so we can test whether registration paperwork is a usable analytical angle. This is distinct from sticker violations: it checks whether the supporting vehicle-status fields are complete enough to trust.

In [ ]:
expiration_status = pd.read_csv(EDA_DIR / 'expiration_status.csv')
expiration_family = pd.read_csv(EDA_DIR / 'expiration_status_family.csv')
expiration_delta = pd.read_csv(EDA_DIR / 'expiration_delta_buckets.csv')
sticker_expiration = pd.read_csv(EDA_DIR / 'sticker_code_expiration_status.csv', dtype={'code': str})
registration_metrics = json.load(open(STORY_DIR / 'registration_metrics.json'))

expiration_status

In [ ]:
fig = px.bar(expiration_status.sort_values('tickets'), x='tickets', y='expiration_status', orientation='h',
             title='Vehicle expiration date status')
fig.update_layout(xaxis_title='Tickets', yaxis_title='')
fig.write_html(STORY_DIR / 'fig38_expiration_status.html')
fig

In [ ]:
expiration_family_plot = expiration_family.merge(
    expiration_family.groupby('family', as_index=False)['tickets'].sum().rename(columns={'tickets': 'family_total'}),
    on='family'
)
expiration_family_plot['share'] = expiration_family_plot['tickets'] / expiration_family_plot['family_total']
family_keep = ['Registration / inspection sticker', 'Street cleaning', 'Meter / paid parking',
               'No standing / no parking', 'School-zone speed', 'Bus lane / bus stop', 'Red light']
expiration_family_plot = expiration_family_plot[expiration_family_plot['family'].isin(family_keep)]

fig = px.bar(expiration_family_plot, x='family', y='share', color='expiration_status', barmode='stack',
             title='Expiration-date status by violation family')
fig.update_layout(xaxis_title='', yaxis_title='Share within family', yaxis_tickformat='.0%', legend_title='Expiration status')
fig.write_html(STORY_DIR / 'fig39_expiration_status_by_family.html')
fig

In [ ]:
delta_order = ['Expired >1y', 'Expired 91d-1y', 'Expired 31-90d', 'Expired 1-30d', 'Expires today',
               'Expires 1-30d', 'Valid 31d-1y', 'Valid 1-10y', 'Valid 10y+']
expiration_delta['expiration_delta_bucket'] = pd.Categorical(expiration_delta['expiration_delta_bucket'], categories=delta_order, ordered=True)
expiration_delta = expiration_delta.sort_values('expiration_delta_bucket')

fig = px.bar(expiration_delta, x='expiration_delta_bucket', y='tickets',
             title='Expiration timing among parseable expiration dates')
fig.update_layout(xaxis_title='', yaxis_title='Tickets')
fig.write_html(STORY_DIR / 'fig40_expiration_delta_buckets.html')
fig

### Registration/Expiration Result

This angle is mostly a data-quality warning. Only about 35.6% of rows have a parseable vehicle expiration date; 64.4% are missing or placeholders. Among parseable dates, about 8.5% are expired before the ticket date. The `Unregistered Vehicle?` field is almost always blank or `0`, so it is not very informative. Registration-status fields should not be the center of the project.

## Updated Suggestion After Registration Analysis

Do not build the final story around vehicle expiration status. Use it only as a caveat for sticker/registration interpretations:

**The ticket code tells us there are sticker/registration violations, but the supporting expiration-date field is too incomplete to reconstruct vehicle registration status reliably.**

This reinforces a broader methodological point: some fields support strong analysis (`Violation Code`, `Issue Date`, `Violation Precinct`, `Registration State`, `Plate Type`), while others are too sparse or placeholder-heavy for headline claims.

## Missing Angle: Legal And Curb-Position Fields

The dataset also contains legal/statutory fields: `Law Section`, `Sub Division`, `Violation Legal Code`, and curb-detail fields such as `Feet From Curb`. This checks whether those columns add a new analytical layer or mainly confirm the violation-family structure we already built.


In [ ]:
legal_quality = pd.read_csv(EDA_DIR / 'legal_field_quality.csv')
law_sections = pd.read_csv(EDA_DIR / 'law_section_totals.csv', dtype={'law_section': str})
law_family = pd.read_csv(EDA_DIR / 'law_section_family.csv', dtype={'law_section': str})
law_channel = pd.read_csv(EDA_DIR / 'law_section_channel.csv', dtype={'law_section': str})
law_subdivision = pd.read_csv(EDA_DIR / 'law_subdivision_totals.csv', dtype={'law_section': str, 'sub_division': str})
legal_code_channel = pd.read_csv(EDA_DIR / 'legal_code_channel.csv', dtype={'violation_legal_code': str})
feet_family = pd.read_csv(EDA_DIR / 'feet_from_curb_family_nonzero_share.csv')
legal_metrics = json.load(open(STORY_DIR / 'legal_metrics.json'))

legal_quality


In [ ]:
fig = px.bar(legal_quality.sort_values('usable_share'), x='usable_share', y='field', orientation='h',
             text=legal_quality.sort_values('usable_share')['usable_share'].map(lambda x: f'{x:.1%}'),
             title='Usability of legal and curb-position fields')
fig.update_layout(xaxis_title='Share of FY2025 rows', yaxis_title='', xaxis_tickformat='.0%')
fig.write_html(STORY_DIR / 'fig42_legal_field_usability.html')
fig


In [ ]:
law_top = law_sections.head(8)['law_section'].tolist()
law_family_plot = law_family[law_family['law_section'].isin(law_top)].copy()
law_family_plot['law_section'] = pd.Categorical(law_family_plot['law_section'], categories=law_top[::-1], ordered=True)

fig = px.bar(law_family_plot, x='tickets', y='law_section', color='violation_family', orientation='h',
             title='Top law sections are tied to different enforcement families')
fig.update_layout(xaxis_title='Tickets', yaxis_title='Law Section', legend_title='Violation family')
fig.write_html(STORY_DIR / 'fig43_law_section_family_mix.html')
fig


In [ ]:
law_channel_plot = law_channel[law_channel['law_section'].isin(law_top)].copy()
law_channel_plot['law_section'] = pd.Categorical(law_channel_plot['law_section'], categories=law_top[::-1], ordered=True)

fig = px.bar(law_channel_plot, x='tickets', y='law_section', color='channel', orientation='h',
             title='Law sections also separate camera-admin and curbside records')
fig.update_layout(xaxis_title='Tickets', yaxis_title='Law Section', legend_title='Channel')
fig.write_html(STORY_DIR / 'fig44_law_section_channel_mix.html')
fig


In [ ]:
legal_code_plot = legal_code_channel[legal_code_channel['violation_legal_code'] != '0'].copy()

fig = px.bar(legal_code_plot, x='tickets', y='violation_legal_code', color='channel', orientation='h',
             title='Violation Legal Code mostly marks precinct-0 camera/admin records')
fig.update_layout(xaxis_title='Tickets', yaxis_title='Violation Legal Code', legend_title='Channel')
fig.write_html(STORY_DIR / 'fig47_legal_code_channel_mix.html')
fig


In [ ]:
combo = law_subdivision.head(15).copy()
combo['law_subdivision'] = combo['law_section'] + ' / ' + combo['sub_division']
combo = combo.iloc[::-1]

fig = px.bar(combo, x='tickets', y='law_subdivision', color='top_family', orientation='h',
             title='Most common law/subdivision combinations')
fig.update_layout(xaxis_title='Tickets', yaxis_title='Law section / subdivision', legend_title='Dominant family')
fig.write_html(STORY_DIR / 'fig45_law_subdivision_combinations.html')
fig


In [ ]:
feet_plot = feet_family[feet_family['tickets'] >= 10_000].sort_values('nonzero_feet_share')

fig = px.bar(feet_plot, x='nonzero_feet_share', y='violation_family', orientation='h',
             text=feet_plot['nonzero_feet_share'].map(lambda x: f'{x:.1%}'),
             title='Feet From Curb is only informative for hydrant tickets')
fig.update_layout(xaxis_title='Rows with Feet From Curb > 0', yaxis_title='', xaxis_tickformat='.0%')
fig.write_html(STORY_DIR / 'fig46_feet_from_curb_family_share.html')
fig


### Legal / Curb-Position Result

`Law Section` and `Sub Division` are almost complete, so they are useful as a structured way to validate the violation-family story. `Law Section 408` covers about 9.75M tickets, while `1180 / B` is the largest law/subdivision combination and exactly matches the 4.80M school-zone speed tickets. `Violation Legal Code` is present in 46.2% of rows and mostly marks precinct-0 camera/admin records, not a general legal-detail field.

The special columns `No Standing or Stopping Violation`, `Hydrant Violation`, and `Double Parking Violation` are blank for every FY2025 row, so they should not be used. `Feet From Curb` is numeric everywhere but non-zero in only 1.7% of rows; nearly all useful signal there belongs to hydrant tickets, where 42.5% have a non-zero distance.


## Updated Suggestion After Legal-Field Analysis

The next non-redundant step should be a summons-record integrity audit: check whether `Summons Number` is unique, whether duplicate summons rows exist, and whether duplicated ticket IDs carry conflicting fields. That will tell us how confidently we can say one row equals one ticket before turning the analysis into a final story.

If the summons audit is clean, the project is ready to shift from exploration to narrative assembly: enforcement systems, geography, timing, money, vehicle profile, repeat exposure, and data-quality caveats.


## Missing Angle: Summons Record Integrity

Before treating row counts as ticket counts, we should verify whether `Summons Number` behaves like a true unique ticket identifier. This checks missing IDs, invalid IDs, duplicated summons numbers, and whether duplicates would create conflicting records.


In [ ]:
summons_quality = pd.read_csv(EDA_DIR / 'summons_quality.csv')
summons_dist = pd.read_csv(EDA_DIR / 'summons_count_distribution.csv')
summons_metrics = json.load(open(STORY_DIR / 'summons_metrics.json'))

summons_quality


In [ ]:
summary = pd.DataFrame({
    'category': ['Unique summons numbers', 'Extra duplicate rows', 'Missing/invalid summons'],
    'rows': [summons_metrics['unique_summons_numbers'], summons_metrics['extra_duplicate_rows'],
             summons_metrics['missing_summons_rows'] + summons_metrics['invalid_summons_rows']]
})

fig = px.bar(summary, x='category', y='rows', text='rows',
             title='Summons Number integrity summary')
fig.update_layout(xaxis_title='', yaxis_title='Rows / IDs')
fig.write_html(STORY_DIR / 'fig48_summons_integrity_summary.html')
fig


In [ ]:
fig = px.bar(summons_dist.head(10).assign(rows_per_summons=lambda d: d['rows_per_summons'].astype(str)),
             x='rows_per_summons', y='summons_numbers',
             title='How many rows share the same Summons Number?')
fig.update_layout(xaxis_title='Rows per summons number', yaxis_title='Summons numbers')
fig.write_html(STORY_DIR / 'fig49_summons_count_distribution.html')
fig


### Summons Integrity Result

This check is clean. All 16.25M FY2025 rows have a valid `Summons Number`, and all 16.25M summons numbers are unique. There are no duplicate summons IDs and no extra duplicate rows, so row counts can be interpreted as summons/ticket counts without needing a duplicate correction.


## Updated Suggestion After Summons Integrity

The dataset is structurally strong enough for a final narrative: one FY2025 row equals one summons. The next useful squeeze is not another data-cleaning check; it is a daily anomaly scan to find unusual high/low enforcement dates that monthly and weekday charts hide.


## Missing Angle: Daily Anomalies And Cutoff Effects

Monthly, weekday, and hourly charts hide exact-date shocks. This section compares each date to the median for the same weekday, then separates likely end-of-file cutoff effects from meaningful high/low enforcement days.


In [ ]:
daily_total = pd.read_csv(EDA_DIR / 'daily_total.csv', parse_dates=['issue_date'])
daily_channel = pd.read_csv(EDA_DIR / 'daily_channel.csv', parse_dates=['issue_date'])
daily_anomalies = pd.read_csv(EDA_DIR / 'daily_total_anomalies_excluding_cutoff.csv', parse_dates=['issue_date'])
family_coverage = pd.read_csv(EDA_DIR / 'daily_family_coverage.csv')
daily_metrics = json.load(open(STORY_DIR / 'daily_anomaly_metrics.json'))

daily_total.tail(10)


In [ ]:
channel_daily = daily_channel.pivot_table(index='issue_date', columns='channel', values='tickets', aggfunc='sum').sort_index().fillna(0)
channel_rolling = channel_daily.rolling(7, min_periods=1).mean().reset_index().melt(
    id_vars='issue_date', var_name='channel', value_name='tickets_7d_avg'
)

fig = px.line(channel_rolling, x='issue_date', y='tickets_7d_avg', color='channel',
              title='Daily ticket volume, 7-day rolling average')
fig.update_layout(xaxis_title='', yaxis_title='Tickets, 7-day average')
fig.write_html(STORY_DIR / 'fig51_daily_channel_rolling_average.html')
fig


In [ ]:
anom_plot = daily_anomalies.copy()
anom_plot['date_label'] = anom_plot['issue_date'].dt.strftime('%Y-%m-%d') + ' (' + anom_plot['weekday'].str[:3] + ')'
anom_plot = anom_plot.sort_values(['anomaly_type', 'ratio_to_weekday_median'])

fig = px.bar(anom_plot, x='ratio_to_weekday_median', y='date_label', color='anomaly_type', orientation='h',
             title='Unusual dates after excluding likely end-of-file cutoff')
fig.update_layout(xaxis_title='Ratio to same-weekday median', yaxis_title='')
fig.write_html(STORY_DIR / 'fig54_daily_total_anomalies_excluding_cutoff.html')
fig


In [ ]:
major_coverage = family_coverage[family_coverage['tickets'] >= 100_000].sort_values('longest_zero_run_days')

fig = px.bar(major_coverage, x='longest_zero_run_days', y='violation_family', orientation='h',
             title='Longest zero-volume runs by violation family')
fig.update_layout(xaxis_title='Longest run of dates with zero tickets', yaxis_title='')
fig.write_html(STORY_DIR / 'fig55_family_zero_run_coverage.html')
fig


In [ ]:
mta_daily = pd.read_csv(EDA_DIR / 'daily_family.csv', parse_dates=['issue_date'])
mta_daily = mta_daily[mta_daily['violation_family'] == 'MTA camera double parking']
all_dates = pd.date_range('2024-07-01', '2025-06-30')
mta_daily = mta_daily.set_index('issue_date')['tickets'].reindex(all_dates).fillna(0).reset_index()
mta_daily.columns = ['issue_date', 'tickets']

fig = px.line(mta_daily, x='issue_date', y='tickets',
              title='MTA camera double-parking records appear only within a bounded date window')
fig.update_layout(xaxis_title='', yaxis_title='Tickets')
fig.write_html(STORY_DIR / 'fig56_mta_camera_double_parking_daily.html')
fig


### Daily-Anomaly Result

The exact-date scan adds two important caveats. First, the last six dates of FY2025 look like a dataset cutoff, not real enforcement behavior: volume collapses starting `2025-06-25`, then `2025-06-29` has only 1 ticket and `2025-06-30` has 18. Treat late-June totals carefully.

After excluding that cutoff window, the highest total-volume outlier is `2024-11-29` with 65,780 tickets, about 1.30x a normal Friday. Low outliers line up with obvious calendar disruptions: `2024-11-28`, `2024-12-25`, `2025-01-01`, `2024-07-04`, and `2025-05-26` are all far below their same-weekday baselines. The dataset alone does not prove the cause, but the date pattern is consistent with holiday effects.

Family coverage also matters. `MTA camera double parking` has no FY2025 records until `2024-08-19` and none after `2025-06-09`, so it should be described as a bounded-window series, not a full-year daily trend.


## Updated Suggestion After Daily-Anomaly Analysis

We have now squeezed the main analytical angles: systems, geography, timing, money, families, vehicles, repeat exposure, observation rules, issuers, legal fields, summons integrity, and exact-date anomalies. The next high-value step is to assemble the final narrative and cut redundant charts.

Recommended final story spine: **NYC FY2025 parking summonses are not one enforcement system. They are a mix of automated/admin programs and curbside officer enforcement, each with different geography, timing, legal structure, fine value, and data-quality limits.**


# The Uneven Curb
## How NYC's Parking-Ticket System Changes Depending on Where You Stand

This project is not trying to prove social inequality from parking-ticket records alone. The dataset does not contain the resident, driver, trip, income, street-use, or exposure denominators needed to make that kind of claim responsibly.

Instead, this project studies uneven enforcement geography inside NYC's FY2025 parking-ticket system. The central idea is that parking enforcement does not behave like one uniform citywide system. It behaves like multiple systems layered together: automated/admin records, real curbside precinct enforcement, different violation families, different timing patterns, and different estimated fine values.

That makes the dataset interesting even without making a direct inequality claim. A city rule system can feel different depending on where and when someone encounters it. The same broad parking-ticket system may appear as school-zone speed cameras in one place, street-cleaning tickets in another, meter enforcement at one time of day, or high-fine hydrant and double-parking tickets in another context.

**Main research question:**

When we separate administrative/automated enforcement from real curbside precinct enforcement, how unevenly does NYC's FY2025 parking-ticket system operate across precincts, violation families, time, and estimated fine value?

**Subquestions:**

1. Which real precincts carry the largest share of curbside tickets?
2. Which violation families dominate different parts of the city?
3. Do ticket counts, timing, and estimated fine value tell the same story, or different stories?


## Where the Analysis Already Stands

The notebook has already completed broad exploratory analysis on the FY2025 NYC parking violations dataset. The raw CSV is about 2.85 GB, with 16,559,243 raw rows and 43 variables. After parsing `Issue Date` and restricting the file to FY2025, the working analysis uses 16,250,291 ticket rows covering `2024-07-01` through `2025-06-30`. The scan found 1,470 invalid dates and 307,482 rows outside the FY2025 window.

The main cleaning decisions are now established. The analysis filters to valid FY2025 issue dates, keeps compact summary tables in `outputs/eda/`, writes story-ready figures and metrics in `outputs/story/`, uses anonymized vehicle and issuer identifiers where needed, and treats incomplete fields as caveats rather than headline evidence. It also flags a likely end-of-file cutoff beginning `2025-06-25`, so the final late-June dates should not be interpreted as normal enforcement behavior.

`Precinct 0` has been separated from real precinct geography. It accounts for 7,524,683 FY2025 rows, about 46.3% of the working dataset, and is dominated by automated/admin-style records such as school-zone speed, bus lane, red-light, and MTA camera violations. Because it is not a mappable precinct polygon, the precinct maps and curbside precinct analysis exclude it and treat it as its own enforcement channel.

Violation-code and violation-family analysis has already been built. The largest family is `School-zone speed` with 4,801,690 tickets, followed by `Bus lane / bus stop` with 1,818,983 tickets. The family grouping separates school-zone speed, bus lane/bus stop, red light, MTA camera double parking, street cleaning, meter/paid parking, no standing/no parking, hydrant, registration/inspection sticker, double parking, and other records.

Borough analysis shows that the citywide story changes by place. Manhattan carries the largest curbside-ticket count, while Staten Island has the highest camera/admin share. Brooklyn, Queens, Bronx, and Staten Island are more dominated by school-zone speed records, while Manhattan has a stronger mix of meter, no-standing, street-cleaning, and other curbside violations.

Precinct analysis has been completed for real, mappable precincts. The notebook maps 78 precincts and 8,724,635 real-precinct tickets. Street cleaning is the dominant family in 44 mapped precincts, while precinct `122` has the strongest specialization pattern, with registration/inspection sticker tickets about 3.68 times more prominent than the mapped-precinct average.

The enforcement-channel analysis separates `Precinct 0 / camera-admin` from `Real precinct / curbside`. This split is now one of the core findings: the two channels have different geographies, different violation mixes, different issuing agencies, and different time patterns.

Timing analysis has been completed by month, weekday, hour, and hour-weekday. Precinct `0` records peak around hour `15`, while real-precinct curbside records peak around hour `9`. Precinct `0` peaks by weekday on Sunday, while real-precinct records peak on Tuesday. The monthly precinct-0 share ranges from about 42.3% to 55.4%.

Fine estimation analysis has been added using the data dictionary fine values. School-zone speed is first by both count and estimated fine value, with an estimated total of about $240.1M. The money ranking differs from the count ranking: no-standing/no-parking, hydrant, double parking, and MTA camera double parking become more important when estimated fine value is considered.

Repeat vehicle-key analysis has been completed without exposing raw plate IDs. There are about 4.05M valid vehicle keys. One-time vehicle keys are 43.4% of keys but only 10.8% of tickets, while repeat vehicle keys account for about 89.2% of tickets. The top 1% of vehicle keys account for about 14.9% of tickets, and the highest anonymous vehicle key has 1,088 tickets.

Street concentration analysis has also been completed. The dataset contains 46,646 street-name labels. The top 100 street labels account for about 22.2% of tickets, and `BROADWAY` alone has 179,263 records. However, 9,103 street labels appear in more than one borough, so street-level findings need careful wording unless more geocoding or block-level cleaning is added.

Issuer analysis has been completed. Issuing agency `T` is the largest source with 7,877,929 tickets and is tied mainly to real-precinct curbside records. Agency `V` is dominated by precinct `0`, reinforcing the camera/admin channel. Valid issuer-code values exist for about 52.5% of rows, and the top 1% of anonymized issuer codes account for about 27.9% of tickets with valid issuer codes.

Daily anomaly analysis has been completed. The strongest high-volume date after comparing to same-weekday baselines is `2024-11-29`, with 65,780 tickets, about 1.30 times a normal Friday. Low-volume dates include `2024-11-28`, `2024-12-25`, `2025-01-01`, `2024-07-04`, and `2025-05-26`, which are consistent with major calendar disruptions. The analysis also found that MTA camera double-parking records only appear from `2024-08-19` to `2025-06-09`, so that family should be treated as a bounded-window series.

The remaining work is not more broad EDA. The remaining work is to turn these findings into a focused story about how NYC's parking-ticket system changes across space, rule type, time, and money.


## The Place That Is Not a Place: Precinct 0

Before making spatial claims, `Violation Precinct = 0` has to be separated from real precinct geography. It is analytically important because it contains a very large share of tickets, but it is not normal neighborhood geography and should not be mapped as if it were a precinct polygon.


In [ ]:
precinct0_metrics = json.load(open(STORY_DIR / 'precinct0_narrative_metrics.json'))
precinct0_family = pd.read_csv(EDA_DIR / 'precinct0_vs_real_precincts_family.csv')
precinct_totals = pd.read_csv(EDA_DIR / 'precincts.csv')

precinct0_summary = pd.DataFrame([
    {
        'group': 'Precinct 0',
        'tickets': precinct0_metrics['precinct0_rows'],
        'share_of_fy2025': precinct0_metrics['precinct0_share'],
        'top_family': precinct0_metrics['top_precinct0_family'],
        'top_family_share': precinct0_metrics['top_precinct0_family_share'],
    },
    {
        'group': 'Real precincts',
        'tickets': precinct0_metrics['real_precinct_rows'],
        'share_of_fy2025': precinct0_metrics['real_precinct_share'],
        'top_family': precinct0_metrics['top_real_family'],
        'top_family_share': precinct0_metrics['top_real_family_share'],
    },
])
precinct0_summary


In [ ]:
precinct0_top_families = (
    precinct0_family[precinct0_family['channel'] == 'Precinct 0']
    .sort_values('tickets', ascending=False)
    [['family', 'tickets', 'share_within_channel']]
    .head(8)
)
precinct0_top_families


In [ ]:
real_top_families = (
    precinct0_family[precinct0_family['channel'] == 'Real precinct']
    .sort_values('tickets', ascending=False)
    [['family', 'tickets', 'share_within_channel']]
    .head(8)
)
real_top_families


In [ ]:
family_order = (
    precinct0_family.groupby('family', as_index=False)['tickets'].sum()
    .sort_values('tickets', ascending=False)['family']
    .tolist()
)

fig = px.bar(
    precinct0_family,
    x='channel_label',
    y='share_within_channel',
    color='family',
    category_orders={'channel_label': ['Precinct 0', 'Real precincts'], 'family': family_order},
    custom_data=['tickets', 'share_within_channel', 'channel_total'],
    title='Precinct 0 is a different enforcement subsystem than real precincts',
    labels={'channel_label': '', 'share_within_channel': 'Share within channel', 'family': 'Violation family'},
)
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>%{legendgroup}<br>Tickets: %{customdata[0]:,}<br>Share in channel: %{customdata[1]:.1%}<br>Channel total: %{customdata[2]:,}<extra></extra>'
)
fig.update_layout(yaxis_tickformat='.0%', legend=dict(orientation='h', y=-0.32), height=620)
fig.write_html(STORY_DIR / 'fig_precinct0_vs_real_precincts.html')
fig


### Interpretation

`Precinct 0` is the largest precinct value in the file: 7,524,683 tickets, or about 46.3% of FY2025 records. That makes it too important to ignore. But it is not a normal geographic precinct. Its violation mix is dominated by school-zone speed, bus lane/bus stop, red-light, and MTA camera-style records, while real precincts are led by street cleaning, meter/paid parking, no-standing/no-parking, hydrant, and registration/inspection sticker violations.

The better interpretation is that `Precinct 0` represents a different subsystem inside the parking-ticket data: administrative, automated, missing, or non-standard precinct assignment patterns depending on the ticket source. It should not be called fake data, and it should not be deleted from the dataset entirely. It should be separated before making spatial claims.

Once `Precinct 0` is separated, the real curbside geography becomes clearer. The map can focus on actual precinct polygons, while the automated/admin channel can be analyzed as its own citywide layer rather than being forced into neighborhood geography.


## The Real Curb Is Not Evenly Shared

After separating `Precinct 0`, the next question is whether real curbside ticketing is evenly spread across actual precinct geography. This section uses only real mappable precincts: `Precinct 0` is excluded, missing/non-standard precinct labels are excluded, and only precinct labels that join to the NYPD precinct GeoJSON are kept.


In [ ]:
real_precinct_concentration = pd.read_csv(EDA_DIR / 'real_precinct_concentration_metrics.csv')
real_precinct_summary = json.load(open(STORY_DIR / 'real_precinct_concentration_summary.json'))

real_precinct_concentration.head(10)


In [ ]:
real_precinct_summary


In [ ]:
import plotly.graph_objects as go

curve = real_precinct_concentration.copy()
curve['cumulative_precinct_share'] = curve['rank'] / len(curve)
curve = pd.concat([
    pd.DataFrame([{
        'precinct': 'start', 'tickets': 0, 'rank': 0,
        'cumulative_precinct_share': 0.0, 'cumulative_ticket_share': 0.0
    }]),
    curve
], ignore_index=True)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=curve['cumulative_precinct_share'],
    y=curve['cumulative_ticket_share'],
    mode='lines+markers',
    name='Real precinct ticket concentration',
    customdata=curve[['precinct', 'rank', 'tickets', 'cumulative_precinct_share', 'cumulative_ticket_share']],
    hovertemplate=(
        'Precinct: %{customdata[0]}<br>'
        'Rank: %{customdata[1]:.0f}<br>'
        'Tickets: %{customdata[2]:,}<br>'
        'Cumulative precinct share: %{customdata[3]:.1%}<br>'
        'Cumulative ticket share: %{customdata[4]:.1%}<extra></extra>'
    ),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines', name='Equal distribution',
    line=dict(color='gray', dash='dash')
))
fig.update_layout(
    title='A small share of precincts carries a large share of curbside tickets',
    xaxis_title='Cumulative share of real precincts',
    yaxis_title='Cumulative share of real-precinct tickets',
    xaxis_tickformat='.0%',
    yaxis_tickformat='.0%',
    height=560,
)
fig.write_html(STORY_DIR / 'fig_real_precinct_concentration_curve.html')
fig


### Interpretation

Real curbside enforcement is concentrated, not evenly distributed. Among 78 mappable real precincts, the top 10% of precincts carry about 27.2% of real-precinct tickets, and the top 25% carry about 49.9%. The Gini coefficient is about 0.37 and the coefficient of variation is about 0.72, which means the real-precinct footprint is meaningfully uneven even after removing `Precinct 0`.

The highest-volume real precinct is precinct `19`, with 447,142 tickets. The bottom nonzero mappable precinct is precinct `22`, with 219 tickets. This large range shows that curbside enforcement is broadly present across the city, but the ticket burden is not evenly shared across precinct polygons.

This is an enforcement-footprint measure. It tells us where tickets are recorded, not whether enforcement is fair or unfair. The metric is not normalized by curb length, meter supply, car ownership, population, traffic volume, land use, or the amount of regulated parking space available in each precinct.


## Different Precincts, Different Parking Rules

Real-precinct ticket totals show where curbside enforcement is concentrated. This section asks a different question: whether the mix of parking rules changes by precinct. A precinct can have moderate total volume but still stand out because one violation family is unusually prominent there.


In [ ]:
real_precinct_family_profile = pd.read_csv(EDA_DIR / 'real_precinct_family_profile.csv')
family_specialization_summary = json.load(open(STORY_DIR / 'real_precinct_family_specialization_summary.json'))

real_precinct_family_profile.head(12)


In [ ]:
interesting_family_examples = (
    real_precinct_family_profile[
        (real_precinct_family_profile['tickets'] >= 1000) &
        (real_precinct_family_profile['family'] != 'Other')
    ]
    .sort_values(['specialization_ratio', 'tickets'], ascending=[False, False])
    .head(15)
)
interesting_family_examples


In [ ]:
import numpy as np
import plotly.graph_objects as go

top_families = family_specialization_summary['top_families_in_heatmap']
precinct_order = (
    real_precinct_family_profile[['precinct', 'precinct_total']]
    .drop_duplicates()
    .sort_values('precinct_total', ascending=False)['precinct']
    .astype(str)
    .tolist()
)
heat = real_precinct_family_profile[real_precinct_family_profile['family'].isin(top_families)].copy()
heat['precinct_label'] = heat['precinct'].astype(str)

z = heat.pivot(index='precinct_label', columns='family', values='specialization_ratio').loc[precinct_order, top_families]
tickets = heat.pivot(index='precinct_label', columns='family', values='tickets').loc[precinct_order, top_families]
share = heat.pivot(index='precinct_label', columns='family', values='family_share_within_precinct').loc[precinct_order, top_families]
city_share = heat.pivot(index='precinct_label', columns='family', values='citywide_family_share_among_real_precincts').loc[precinct_order, top_families]
custom = np.stack([tickets.values, share.values, city_share.values, z.values], axis=-1)

color_max = max(2.5, float(np.nanpercentile(z.values, 97)))
color_min = max(0, 2 - color_max)
fig = go.Figure(data=go.Heatmap(
    z=z.values,
    x=top_families,
    y=precinct_order,
    colorscale='RdBu_r',
    zmid=1,
    zmin=color_min,
    zmax=color_max,
    colorbar=dict(title='Specialization ratio'),
    customdata=custom,
    hovertemplate=(
        'Precinct: %{y}<br>'
        'Violation family: %{x}<br>'
        'Tickets: %{customdata[0]:,}<br>'
        'Share within precinct: %{customdata[1]:.1%}<br>'
        'Citywide share: %{customdata[2]:.1%}<br>'
        'Specialization ratio: %{customdata[3]:.2f}<extra></extra>'
    ),
))
fig.update_layout(
    title='Different precincts specialize in different parking rules',
    xaxis_title='Violation family',
    yaxis_title='Real precincts ordered by total tickets',
    height=1050,
)
fig.update_xaxes(tickangle=-35)
fig.write_html(STORY_DIR / 'fig_real_precinct_family_specialization_heatmap.html')
fig


### Interpretation

The specialization ratio compares a family's share inside one precinct with that family's share across all real mappable precincts. A value above `1` means the family is more prominent in that precinct than it is citywide among real precincts. A value below `1` means the family is less prominent there.

Several precinct/family combinations stand out:

1. Precinct `123` is highly specialized in `Registration / inspection sticker` tickets: 8,190 tickets, 56.1% of that precinct's profile, about 5.54 times the real-precinct citywide share.
2. Precinct `76` stands out for `Street cleaning`: 17,637 tickets, 52.1% of the precinct profile, about 2.51 times the citywide real-precinct share.
3. Precinct `111` is concentrated in `Meter / paid parking`: 16,684 tickets, 42.3% of its profile, about 2.52 times the citywide share.
4. Precinct `34` is more prominent for `Double parking`: 11,697 tickets, 11.5% of its profile, about 3.00 times the citywide share.
5. Precinct `62` stands out for `Hydrant`: 24,828 tickets, 18.5% of its profile, about 2.45 times the citywide share.

These are rule-profile differences, not proof of targeting or unfairness. The heatmap shows that real precincts are associated with different mixes of parking rules: some are more street-cleaning oriented, some are more meter-oriented, some are more sticker-oriented, and some stand out for hydrant or double-parking records.


## Mapping the Curbside Footprint

The previous sections separated `Precinct 0` from real precincts and showed that real curbside ticketing is concentrated. This section turns that result into the main geographic visualization, using only precinct polygons that can be joined to the NYPD precinct GeoJSON.


In [ ]:
geojson_path = OUTPUT_DIR / 'police_precincts.geojson'
with open(geojson_path) as f:
    precinct_geojson = json.load(f)
for feature in precinct_geojson['features']:
    feature['id'] = str(feature['properties']['precinct'])

real_precinct_map_profile = pd.read_csv(EDA_DIR / 'real_precinct_map_profile.csv', dtype={'precinct': str})
real_precinct_map_summary = json.load(open(STORY_DIR / 'real_precinct_map_summary.json'))

real_precinct_map_profile.head(10)


In [ ]:
fig = px.choropleth_mapbox(
    real_precinct_map_profile,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='id',
    color='share_of_real_precinct_tickets',
    color_continuous_scale='YlOrRd',
    hover_name='precinct',
    hover_data={
        'precinct': False,
        'tickets': ':,',
        'share_of_real_precinct_tickets': ':.2%',
        'dominant_violation_family': True,
        'average_estimated_fine': ':$.2f',
        'top_3_families': True,
    },
    mapbox_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.72,
    title='Real-precinct share of NYC curbside parking tickets',
    labels={
        'share_of_real_precinct_tickets': 'Share of real-precinct tickets',
        'tickets': 'Ticket count',
        'dominant_violation_family': 'Dominant family',
        'average_estimated_fine': 'Avg. estimated fine',
        'top_3_families': 'Top 3 families',
    },
)
fig.update_layout(
    margin=dict(l=0, r=0, t=55, b=0),
    coloraxis_colorbar=dict(tickformat='.1%', title='Ticket share'),
)
fig.write_html(STORY_DIR / 'fig_real_precinct_ticket_share_map.html')
fig


In [ ]:
fig = px.choropleth_mapbox(
    real_precinct_map_profile,
    geojson=precinct_geojson,
    locations='precinct',
    featureidkey='id',
    color='dominant_violation_family',
    hover_name='precinct',
    hover_data={
        'precinct': False,
        'tickets': ':,',
        'share_of_real_precinct_tickets': ':.2%',
        'dominant_family_share': ':.1%',
        'top_3_families': True,
        'average_estimated_fine': ':$.2f',
    },
    mapbox_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.75,
    title='Dominant parking-rule family by real precinct',
    labels={
        'dominant_violation_family': 'Dominant family',
        'tickets': 'Ticket count',
        'share_of_real_precinct_tickets': 'Share of real-precinct tickets',
        'dominant_family_share': 'Dominant family share',
        'top_3_families': 'Top 3 families',
        'average_estimated_fine': 'Avg. estimated fine',
    },
)
fig.update_layout(margin=dict(l=0, r=0, t=55, b=0), legend=dict(orientation='h', y=-0.05))
fig.write_html(STORY_DIR / 'fig_dominant_family_by_precinct_map.html')
fig


### Interpretation

The curbside ticket geography is not uniform. The ticket-share map shows that some real precincts carry much larger shares of the real-precinct ticket footprint than others. The dominant-family map adds a second layer: high-volume precincts do not all have the same rule profile. Some precincts are more shaped by street cleaning, while others are more shaped by no-standing/no-parking, meter/paid-parking, registration/inspection sticker, or other families.

These raw precinct maps are not risk maps. They show where tickets were issued, not where illegal parking was most common. They are not adjusted for curb length, meter density, population, traffic volume, car ownership, land use, or the amount of regulated parking supply. The maps are best read as an enforcement-footprint view of the real curbside subsystem, after removing `Precinct 0` and leaving unmappable records out of the geography.


## Tickets Are Counted Once, But They Do Not Cost the Same

Ticket counts measure how often summonses are issued, but different violation families have different fine amounts. This section compares real-precinct ticket volume with estimated fine value using the same family-level fine-estimation logic developed earlier in the notebook.


In [ ]:
real_precinct_fine_value = pd.read_csv(EDA_DIR / 'real_precinct_estimated_fine_value.csv', dtype={'precinct': str})
real_precinct_fine_summary = json.load(open(STORY_DIR / 'real_precinct_estimated_fine_value_summary.json'))

real_precinct_fine_value.head(12)


In [ ]:
rank_shift_examples = pd.concat([
    real_precinct_fine_value.sort_values('rank_shift', ascending=False).head(6),
    real_precinct_fine_value.sort_values('rank_shift', ascending=True).head(6),
])
rank_shift_examples[[
    'precinct', 'ticket_count', 'estimated_total_fine_value', 'average_estimated_fine_per_ticket',
    'rank_by_ticket_count', 'rank_by_estimated_fine_value', 'rank_shift', 'dominant_violation_family'
]]


In [ ]:
label_precincts = set(real_precinct_fine_value.head(8)['precinct'])
eligible_shift = real_precinct_fine_value[real_precinct_fine_value['ticket_count'] >= 25_000]
label_precincts.update(eligible_shift.sort_values('rank_shift', ascending=False).head(4)['precinct'])
label_precincts.update(eligible_shift.sort_values('rank_shift', ascending=True).head(4)['precinct'])
plot_df = real_precinct_fine_value.copy()
plot_df['label'] = plot_df['precinct'].where(plot_df['precinct'].isin(label_precincts), '')

fig = px.scatter(
    plot_df,
    x='ticket_count',
    y='estimated_total_fine_value',
    color='dominant_violation_family',
    size='average_estimated_fine_per_ticket',
    text='label',
    hover_name='precinct',
    hover_data={
        'precinct': False,
        'ticket_count': ':,',
        'estimated_total_fine_value': ':$,.0f',
        'average_estimated_fine_per_ticket': ':$.2f',
        'rank_by_ticket_count': True,
        'rank_by_estimated_fine_value': True,
        'rank_shift': True,
        'dominant_violation_family': True,
        'label': False,
    },
    title='Ticket counts and estimated fine value tell related but not identical stories',
    labels={
        'ticket_count': 'Ticket count',
        'estimated_total_fine_value': 'Estimated total fine value',
        'average_estimated_fine_per_ticket': 'Average estimated fine',
        'dominant_violation_family': 'Dominant violation family',
    },
)
fig.update_traces(textposition='top center')
fig.update_layout(
    yaxis_tickprefix='$',
    yaxis_tickformat=',.0f',
    xaxis_tickformat=',',
    height=650,
    legend=dict(orientation='h', y=-0.25),
)
fig.write_html(STORY_DIR / 'fig_real_precinct_count_vs_estimated_fine_value.html')
fig


### Interpretation

High-ticket precincts are generally also high estimated-fine-value precincts. Precinct `19` ranks first by both measures, with 447,142 tickets and about $31.6M in estimated fine value. Precinct `14` and precinct `13` also remain near the top by both count and estimated fine value.

The rankings are not identical, because the violation mix changes the average estimated fine per ticket. Positive rank shift means a precinct ranks higher by estimated fine value than by ticket count. Precinct `33` rises from ticket-count rank 47 to estimated-fine-value rank 38, and precinct `17` rises from rank 22 to rank 15. These precincts have a higher-value mix than their raw counts alone suggest.

Other precincts move the opposite direction. Precinct `78` falls from ticket-count rank 40 to estimated-fine-value rank 50, and precinct `112` falls from rank 12 to rank 20. Their ticket counts are substantial, but their average estimated fine per ticket is lower, so the estimated fine-value footprint is smaller than the count ranking suggests.

These figures are **estimated fine value**, not actual collected revenue. The dataset contains issued tickets and violation codes; it does not contain payment outcomes, dismissals, reductions, late fees, or collection status.


## The Uneven Curb Has a Clock

The curbside system is not only spatial. Different parking rules also appear at different times of day. This section compares hourly profiles by violation family, using each hour's share of that family's valid-time tickets.


In [ ]:
violation_family_hourly = pd.read_csv(EDA_DIR / 'violation_family_hourly_profiles.csv')
hourly_profile_summary = json.load(open(STORY_DIR / 'violation_family_hourly_profiles_summary.json'))

pd.read_csv(EDA_DIR / 'violation_family_hourly_peak_summary.csv')


In [ ]:
import numpy as np
import plotly.graph_objects as go

default_visible = hourly_profile_summary['default_visible_families']
families = hourly_profile_summary['families_in_figure']

fig = go.Figure()
for family in families:
    d = violation_family_hourly[violation_family_hourly['family'] == family].sort_values('hour')
    fig.add_trace(go.Scatter(
        x=d['hour'],
        y=d['share_of_family_daily_tickets'],
        mode='lines+markers',
        name=family,
        visible=True if family in default_visible else 'legendonly',
        customdata=np.stack([d['tickets'], d['share_of_family_daily_tickets']], axis=-1),
        hovertemplate=(
            'Family: ' + family + '<br>'
            'Hour: %{x}:00<br>'
            'Ticket count: %{customdata[0]:,}<br>'
            'Share of family daily tickets: %{customdata[1]:.1%}<extra></extra>'
        ),
    ))
fig.update_layout(
    title='Different parking rules have different daily rhythms',
    xaxis=dict(title='Hour of day', tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(title="Share of each family's valid-time tickets", tickformat='.0%'),
    height=650,
    legend=dict(orientation='h', y=-0.25),
)
fig.write_html(STORY_DIR / 'fig_violation_family_hourly_profiles.html')
fig


### Interpretation

Different rule families peak at different times. `Hydrant` tickets peak around hour `6`, `Registration / inspection sticker` peaks around hour `8`, and `Street cleaning` peaks sharply around hour `9`. `Meter / paid parking`, `No standing / no parking`, and `Double parking` peak later, around hour `13`.

This means the city's parking-ticket system is both spatial and temporal. The maps show where different parts of the curbside system are recorded; the hourly profiles show that different rules also have different daily rhythms.

These times describe when tickets are issued in the dataset, not necessarily the exact moment when a violation first occurred. That distinction matters especially for camera/admin-style records or records that may contain administrative timestamp artifacts. The missing-hour rate is small, about 0.19% of FY2025 rows, but issue time should still be interpreted as a recorded enforcement timestamp rather than a complete behavioral clock.


## Final Narrative Spine

**Title:**  
The Uneven Curb

**Subtitle:**  
How NYC's Parking-Ticket System Changes Depending on Where You Stand

### Story Sections

1. **One city, several ticket systems**  
   NYC parking tickets look like one system, but the data separates into automated/admin and real curbside enforcement. The opening should establish the scale of the dataset and the main split between `Precinct 0 / camera-admin` and `Real precinct / curbside` records.

2. **The place that is not a place**  
   Precinct `0` is large and important, but it is not normal geography. It should be treated as a separate enforcement subsystem rather than mapped as a neighborhood or precinct polygon.

3. **The real curb is uneven**  
   Among real mappable precincts, tickets are concentrated rather than evenly spread. The top precincts carry a disproportionate share of real-precinct tickets, so the curbside footprint is uneven even after removing Precinct `0`.

4. **Different precincts, different parking rules**  
   Real precincts do not only differ in volume. They also differ in rule mix. Some places are dominated by street cleaning, others by meters, hydrants, double parking, no-standing/no-parking, or sticker-related tickets.

5. **Tickets and money tell slightly different stories**  
   Estimated fine value adds a second layer beyond raw ticket counts. High-ticket precincts are often high estimated-fine-value precincts, but rank shifts show that violation mix changes the estimated financial footprint.

6. **The curb also has a clock**  
   Violation families follow different daily rhythms. Street cleaning, meters, hydrants, double parking, no-standing/no-parking, and registration/inspection sticker tickets peak at different times of day.

7. **What this proves and what it does not**  
   This project proves uneven enforcement geography in issued tickets. It does not prove demographic inequality, targeting, or causal unfairness. The dataset records issued summonses, not exposure, curb supply, traffic volume, population, car ownership, payment outcomes, dismissals, or the true underlying rate of illegal parking.

### Recommended Website Figures

- `fig_precinct0_vs_real_precincts.html`: establishes why Precinct `0` must be separated.
- `fig_real_precinct_ticket_share_map.html`: primary map showing the real curbside ticket footprint.
- `fig_dominant_family_by_precinct_map.html`: map showing how dominant rule families change by precinct.
- `fig_real_precinct_concentration_curve.html`: compact evidence that real-precinct tickets are concentrated.
- `fig_real_precinct_count_vs_estimated_fine_value.html`: shows that count and estimated fine value are related but not identical.
- `fig_violation_family_hourly_profiles.html`: shows the temporal layer of the curbside system.

### Recommended Explainer-Notebook-Only Figures

- `fig42_legal_field_usability.html`: useful methodological check, but too detailed for the main website.
- `fig48_summons_integrity_summary.html` and `fig49_summons_count_distribution.html`: important data-integrity evidence, best kept in the notebook.
- `fig54_daily_total_anomalies_excluding_cutoff.html`: useful caveat about exact-date anomalies and the late-June cutoff.
- `fig25_repeat_vehicle_distribution.html`, `fig26_tickets_per_vehicle_state_group.html`, and `fig27_tickets_per_vehicle_plate_class.html`: interesting supporting analysis, but not central to the final story spine.
- `fig33_issuing_agency_totals.html`, `fig34_agency_channel_split.html`, and `fig35_agency_family_mix.html`: useful evidence for the enforcement-apparatus layer, but likely too much for the main website.

### Recommended Interactive Figure

Use `fig_real_precinct_ticket_share_map.html` as the main interactive figure. It lets the reader inspect each real precinct, including ticket count, share of real-precinct tickets, dominant violation family, top family mix, and average estimated fine. This is the clearest interactive expression of the project's main claim.

### Recommended Static Summary Figure

Use a simplified static version of `fig_precinct0_vs_real_precincts.html` or `fig_real_precinct_concentration_curve.html` as the static summary figure. The first explains the core split between systems; the second summarizes the main curbside concentration result in one visual. If only one static figure is allowed, use the Precinct `0` vs real-precinct comparison because it introduces the key methodological decision behind the whole project.
